# 📗 오픈소스 LLM 로컬 실행: 오픈 웨이트 LLM을 고르고 직접 실행하기

지금까지 LLM은 OpenAI API 단원처럼 API로 회사 서버의 모델을 불렀습니다. 이 교안은 최신 오픈 웨이트 LLM을 개발 주체·나라·크기·특징과 벤치마크로 정리하고, 조건에 맞는 모델을 골라 Ollama와 Transformers로 직접 실행합니다. 실습 모델은 Qwen3.5-4B입니다.

<img src="images/교안/20_lesson01_goals.png" width="1050" alt="제목은 '이번 교안의 핵심: 가중치 파일을 골라 내 GPU에서 실행하기'입니다. 1 가중치 파일 고르기: 모델 파일 카드 네 장을 '라이선스'와 '24GB에 들어감' 두 칸으로 거릅니다. EXAONE 4.0 1.2B는 라이선스 X와 24GB 체크, Qwen3.5-4B는 둘 다 체크(초록 테두리), Gemma 4 31B와 Qwen3.5-397B-A17B는 라이선스 체크와 24GB X입니다. 아래 메모는 'BF16 가중치: 10억 개당 약 1.86GiB'입니다. 2 내 GPU에 올리기: 초록 카드에서 화살표가 'GPU 메모리 24GB' 통으로 들어가 아래쪽을 'Qwen3.5-4B 가중치 8.7GiB'가 채우고, 그 위에 'KV 캐시 · 생성 여유' 칸이 있습니다. 통 옆에 자물쇠와 '데이터가 GPU 밖으로 나가지 않음'. 3 두 가지 방법으로 실행: GPU에서 갈라진 화살표가 Ollama(GGUF · 4비트 · 명령 한 줄)와 Transformers(safetensors · BF16 · generate())로 가고, Transformers 옆 '다음 토큰 확률' 막대 셋 아래에 'greedy: 가장 높은 확률'과 '샘플링: 확률대로 뽑기'가 있습니다. 아래 결론 띠: '조건으로 가중치 파일을 고르고, 같은 파일을 내 GPU에서 두 가지 방법으로 실행'.">

**학습 목표**

1. **최신 오픈 웨이트 LLM을 개발 주체·나라, 파라미터 크기, 구조(dense·MoE)와 특징으로 분류합니다.**
2. **라이선스·벤치마크·VRAM으로 후보 테이블을 만들고, 조건에 맞는 모델을 고릅니다.**
3. **Ollama로 모델을 받아 대화하고 추론 모드를 켜고 끄며, LangChain `ChatOllama`로 구조화된 출력·도구 호출·이미지 입력을 사용합니다.**
4. **Transformers로 모델을 불러와 chat template을 거쳐 답을 생성합니다.**
5. **`generation_config`의 기본값을 확인하고 greedy와 샘플링의 차이를 설명합니다.**

**먼저 알아 둘 단위**

1B는 파라미터 10억 개입니다. 모델의 메모리는 파라미터 수 x 숫자 하나를 담는 바이트 수(정밀도)입니다.

- **정밀도(자료형, dtype):** 숫자 하나를 담는 비트 수와 형식입니다.

| 정밀도 | 파라미터 1개 | 1B당 메모리 |
|---|---|---|
| FP32 | 4바이트 | 40억 바이트(약 3.73GiB) |
| BF16 | 2바이트 | 약 1.86GiB |
| INT8 | 1바이트 | 약 0.93GiB |
| INT4(4비트) | 0.5바이트 | 약 0.47GiB |

- **GiB:** 1024³ 바이트(약 10.7억 바이트)입니다. 디스크와 파일 크기에 흔히 쓰는 GB(1000³ 바이트)보다 약 7% 큽니다. GPU 사양의 GB(24GB, 32GB)는 실제로는 GiB에 가까운 값입니다.
- **예:** 실습 모델 Qwen3.5-4B의 글 생성 부분(4.21B)은 FP32 15.7GiB, BF16 7.8GiB, 4비트 약 2.0GiB입니다. 생성할 때는 여기에 약 20%를 더해 어림합니다(교안 02 2-4).

## 1. 실습 준비

### 1-1. 준비 셀

Ollama 설치와 서버 켜기는 이 단원 폴더의 `설치_가이드.md`를 따릅니다.

**장비:** 이 교안은 GPU 메모리 24GB 파드를 기준으로 쓰였고, 여유가 더 있는 32GB 파드에서도 그대로 실행됩니다.

| GPU 메모리 | RunPod GPU 예(시간당 요금) | BF16으로 실행하는 크기(생성 여유 20% 포함) |
|---|---|---|
| 24GB(이 교안 기준) | RTX A5000 약 $0.27(빈 자리가 있을 때), RTX PRO 4000 약 $0.57, RTX 4090 약 $0.89 | 약 10B까지 |
| 32GB | RTX 5090 약 $1.19 | 약 14B까지 |

- **요금:** 시기와 빈 자리에 따라 바뀌므로 파드를 만들 때 RunPod 화면에서 확인합니다.

**미리 내려받기:** 실습에서 쓸 모델(Hugging Face 원본, Ollama `qwen3.5:4b-q4_K_M`)을 먼저 받아 둡니다. 처음 한 번만 오래 걸리므로 실행해 두고 2·3절을 읽습니다. Hugging Face 모델은 Hugging Face 캐시 폴더에, Ollama 모델(GGUF 파일)은 Ollama 서버의 모델 폴더에 따로 저장됩니다. Ollama 태그 이름은 ollama.com/search에서 찾습니다(4절의 라이브러리 화면).

- **GGUF:** llama.cpp(C++로 만든 LLM 실행 프로그램)·Ollama가 쓰는 모델 파일 형식으로, 양자화한 가중치와 토크나이저·설정을 파일 하나에 담습니다.
- **연결 오류가 나면:** `Failed to connect to Ollama`는 Ollama 서버가 꺼져 있다는 뜻입니다. 파드에서는 설치 가이드 2절의 `ollama serve`를 다시 실행하고, 내 컴퓨터에서는 Ollama 앱을 켭니다.
- **`HF_TOKEN` 경고:** 로그인하지 않고 공개 모델을 받는다는 안내라 그대로 진행합니다.

In [ ]:
import asyncio

from huggingface_hub import snapshot_download
import ollama

# 실습에서 쓸 모델을 미리 내려받습니다. 처음 한 번만 오래 걸리고, 다음부터는 이미 받은 파일을 확인만 합니다
# Transformers용 원본 모델(BF16, 약 9GB): Hugging Face 캐시 폴더(기본 ~/.cache/huggingface/hub)에 저장됩니다
# snapshot_download는 일반 함수라서 asyncio.to_thread로 감싸야 다른 다운로드와 함께 기다릴 수 있습니다
tasks = [asyncio.to_thread(snapshot_download, "Qwen/Qwen3.5-4B")]

# Ollama용 모델(GGUF 파일): pull 요청을 받은 Ollama 서버가 내려받아 서버의 모델 폴더(기본 ~/.ollama/models)에 보관합니다
# 태그 이름은 ollama.com/search 에서 모델을 검색하고 모델 페이지의 Tags 탭에서 고릅니다
ollama_client = ollama.AsyncClient()
ollama_tags = ["qwen3.5:4b-q4_K_M"]
for tag in ollama_tags:
    tasks.append(ollama_client.pull(tag))

# 두 다운로드는 받는 곳과 저장 위치가 달라 서로 기다릴 필요가 없으므로 동시에 받고, 결과는 tasks 순서대로 받습니다
results = await asyncio.gather(*tasks)
model_dir = results[0]
print("Hugging Face 모델:", model_dir)
print("Ollama 모델:", ollama_tags)

계산 장치를 고르는 코드는 지난 단원과 같습니다.

In [ ]:
import pandas as pd

import torch

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다 (지난 단원과 같은 코드)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블에서 긴 답이 잘리지 않도록 열 너비 제한을 풉니다
pd.set_option("display.max_colwidth", None)

## 2. 오픈 웨이트 LLM

### 2-1. 로컬 실행이 필요한 때

- **오픈 웨이트 LLM(open-weight LLM):** 가중치 파일을 내려받아 내 컴퓨터에서 직접 실행할 수 있게 공개한 LLM입니다. 흔히 오픈소스 LLM이라고 부르지만, 쓰는 조건은 모델마다 라이선스로 따로 정합니다(3-2).
- **가중치(weight):** 학습으로 정해진 파라미터 가운데 층의 행렬 값입니다. LLM 파라미터는 대부분 가중치라서 모델 파일을 흔히 가중치 파일이라고 부릅니다.

텍스트 임베딩 단원에서 로컬 모델과 API를 비교한 기준을 LLM에 그대로 적용합니다.

| 기준 | API 호출 | 로컬 실행(이 교안) |
|---|---|---|
| 비용 | 보낸 토큰과 받은 토큰만큼 계속 과금 | GPU를 켠 시간만큼 과금(RunPod 시간당 요금) |
| 개인정보 | 질문과 문서가 외부 서버로 나감 | 데이터가 내 GPU 밖으로 나가지 않음 |
| 튜닝 | 제공 회사가 허용한 방식만 가능 | 가중치를 우리 데이터로 직접 학습시킬 수 있음 |
| 준비 | API 키만 있으면 바로 사용 | 모델 고르기·내려받기·GPU 준비가 필요 |

### 2-2. 개발 주체와 나라

오픈 웨이트 LLM은 미국과 중국의 기업이 가장 많이 공개하고, 유럽과 한국도 자기 계열을 냅니다. 아래 테이블과 그림은 Hugging Face 모델 카드 기준입니다.

| 나라 | 개발 주체(계열) |
|---|---|
| 미국 | Meta(Muse, Llama), Google(Gemma), OpenAI(gpt-oss), Microsoft(Phi), NVIDIA(Nemotron), IBM(Granite), AI2(OLMo), Hugging Face(SmolLM) |
| 중국 | 알리바바(Qwen), DeepSeek, 문샷 AI(Kimi), Z.ai(GLM), MiniMax |
| 유럽 | Mistral AI(프랑스, Mistral·Ministral), Swiss AI(스위스, Apertus) |
| 한국 | LG AI연구원(EXAONE), SK텔레콤(A.X), 업스테이지(Solar), 네이버클라우드(HyperCLOVA X SEED), 카카오(Kanana), KT(Mi:dm) |
| 그 밖 | TII(아랍에미리트, Falcon) |

<img src="images/교안/21_model_landscape.png" width="1050" alt="제목은 '오픈 웨이트 LLM 지형: 공개 시기와 전체 파라미터 수'입니다. 가로축은 공개 시기(2024.07~2026.10), 세로축은 전체 파라미터 수(로그 눈금, 1B·10B·100B·1T). 점 색은 나라(미국 파랑, 중국 빨강, 유럽 보라, 한국 초록, 그 밖 회색), 동그라미는 dense, 마름모는 MoE입니다. 1B 근처: Qwen2.5-0.5B, Qwen3.5-0.8B, EXAONE 4.0 1.2B. 수 B: Midm 2.0 Mini, SmolLM3(미국), Phi-4-mini, Ministral 3 3B, Gemma 4 E2B, Falcon-H1R 7B, Apertus 8B. 수십 B: gpt-oss-20b(MoE), HyperCLOVA X SEED 32B, Kanana-2 30B-A3B(MoE), Qwen3.5-35B-A3B(MoE), Gemma 4 31B, EXAONE 4.5, Qwen3.8-27B, Muse Glimmer. 100B 이상은 모두 MoE: Llama 4 Scout, gpt-oss-120b, Mistral Small 4, Solar Open 2, DeepSeek V4 Flash, Motif 3, DeepSeek V3.2, A.X K2, K-EXAONE 2.0, GLM-5.3, Muse Spark, Kimi K3.">

**그림 읽기**
- **크기:** 최근 2년 사이에 1B 아래(Qwen2.5-0.5B)부터 2.8T(Kimi K3)까지 공개되었습니다. 한 계열이 여러 크기를 함께 내서 사용자가 장비에 맞춰 고르게 합니다.
- **구조:** 그림에서 100B를 넘는 모델은 모두 MoE(2-3)입니다.
- **한국:** 정부의 독자 AI 파운데이션 모델 사업의 결과물로 LG(K-EXAONE)·SK텔레콤(A.X K)·업스테이지(Solar Open)가 수백 B MoE를 공개했습니다.

### 2-3. MoE: 전체 파라미터와 활성 파라미터

- **dense 모델:** 토큰 하나를 계산할 때 모든 파라미터를 씁니다. 트랜스포머 단원에서 만든 블록이 이 구조입니다.
- **MoE(Mixture of Experts, 전문가 혼합):** 블록의 FFN을 여러 개의 전문가(expert) FFN으로 나누고, 라우터(router)가 토큰마다 몇 개의 전문가만 골라 계산하는 구조입니다. 목적은 모델의 전체 크기(담는 지식)는 키우면서 토큰 하나의 계산량은 작게 유지하는 것입니다.
- **전체 파라미터와 활성 파라미터(active parameters):** 전체는 모든 전문가를 합친 수이고, 활성은 토큰 하나를 계산할 때 실제로 쓰는 수입니다. 이름의 `35B-A3B`는 전체 35B, 활성 3B라는 뜻입니다.
- **메모리와 속도:** 어떤 전문가가 뽑힐지 미리 모르므로 GPU 메모리에는 전체 파라미터를 모두 올려야 합니다. 생성 속도는 활성 파라미터 수에 가깝습니다.

<img src="images/교안/22_dense_vs_moe.png" width="1050" alt="제목은 'dense와 MoE: 토큰 하나가 계산에 쓰는 파라미터'입니다. 왼쪽 판 dense: 토큰이 주황으로 칠한 FFN 전체로 들어감, 모든 파라미터를 계산. 오른쪽 판 MoE: 토큰이 라우터로 들어가고 라우터가 전문가 8개 중 2개(주황)만 고름, 고른 전문가만 계산. 결론: GPU 메모리는 전체 파라미터, 계산량은 활성 파라미터 기준.">

Qwen3.5-35B-A3B의 설정 파일로 전문가 수를 확인합니다.

- **`AutoConfig.from_pretrained(저장소 이름)`:** 가중치 없이 설정 파일(`config.json`)만 내려받아 읽습니다. 텍스트 임베딩 단원에서 `hidden_size`로 임베딩 차원을 확인한 그 파일입니다.
- **`text_config`:** 이 모델은 이미지 입력도 받는 모델이라 글을 다루는 부분의 설정이 `text_config` 안에 있습니다.

In [ ]:
from transformers import AutoConfig

# MoE 모델의 설정 파일(config.json)만 내려받습니다
moe_config = AutoConfig.from_pretrained("Qwen/Qwen3.5-35B-A3B")

# 층마다 전문가 수와, 토큰 하나가 거치는 전문가 수
moe_text_config = moe_config.text_config
print("층 수:", moe_text_config.num_hidden_layers)
print("층마다 전문가 수:", moe_text_config.num_experts)
print("토큰마다 고르는 전문가 수:", moe_text_config.num_experts_per_tok)

**결과 읽기**
- **전문가 256개 중 8개:** 토큰 하나는 층마다 라우터가 고른 전문가 8개(약 3%)와 모든 토큰이 함께 쓰는 공유 전문가(shared expert) 1개만 거칩니다. 그래서 전체 35B 가운데 약 3B만 계산합니다(어텐션 등 모든 토큰이 함께 쓰는 층 포함).
- **메모리:** 전체 파라미터(Hub 기준 359.5억 개)를 BF16으로 올리면 약 67GiB라 24GB·32GB GPU에는 들어가지 않습니다.

### 2-4. 최신 모델의 특징

- **추론 모드(thinking mode):** 최종 답 전에 풀이 과정을 토큰으로 먼저 생성하는 방식입니다. 수학·코드처럼 여러 단계를 거치는 질문의 정확도를 높이는 것이 목적이고, 대신 생성 토큰과 시간이 늘어납니다. 추론 모드를 제공하는 방식은 두 가지입니다.
  - **두 모델로 따로 내기:** 바로 답하는 Instruct판과 늘 풀이부터 쓰는 Thinking판을 별개의 모델 파일로 냅니다(예: `Qwen3-4B-Instruct-2507`과 `Qwen3-4B-Thinking-2507`). 추론 여부를 바꾸려면 다른 모델을 불러옵니다.
  - **하이브리드 모델:** 한 모델이 두 방식을 모두 학습해, 요청마다 설정 하나로 추론을 켜거나 끕니다. Qwen3.5-4B가 이 방식이고 기본은 켜짐입니다. Ollama는 `think`(4-3), Transformers는 `enable_thinking`(5-2)으로 바꿉니다.
- **멀티모달(multimodal):** 글 말고도 이미지·음성을 입력으로 받습니다. 불러오는 클래스와 입력 준비 방법이 글만 받는 모델(`AutoModelForCausalLM`)과 다릅니다.
- **긴 컨텍스트(long context):** 한 번에 넣을 수 있는 토큰 수가 128K~1M까지 늘었습니다. 입력이 길수록 생성할 때 메모리가 더 듭니다(5절).
- **도구 호출(tool calling):** 모델이 답 대신 "이 함수를 이 인자로 불러 달라"는 요청을 만들도록 학습되어 있습니다(4-5).
- **한국어:** 한국 모델은 한국어 데이터 비중을 높여 학습합니다. 다른 모델은 모델 카드의 지원 언어 목록에 한국어가 있는지 확인합니다.

<img src="images/교안/23_thinking_mode.png" width="1050" alt="제목은 '추론 모드: 답 전에 풀이 과정을 먼저 생성'입니다. 직원이 '회의실 3곳을 4팀에 어떻게 배정할까요?'라고 묻고 질문이 모델로 들어갑니다. enable_thinking=True: 회색 '<think> 풀이 과정 </think>' 뒤에 주황 '답', 결과 '여러 단계 질문에 정확'과 '토큰과 시간이 늘어남'. enable_thinking=False: 주황 '답'만, 결과 '빠르고 짧음'.">

GPU 한 장에 BF16으로 들어가는 작은 모델의 특징입니다(모델 카드와 `config.json` 기준, 컨텍스트의 K는 1,024 토큰).

| 모델 | 파라미터 | 추론 모드 | 입력 | 컨텍스트 |
|---|---|---|---|---|
| Qwen3.5-4B(이 교안) | 4.66B(글 생성 4.21B) | 켜고 끔(기본 켬) | 글·이미지 | 256K |
| Qwen3.5-0.8B | 0.87B | 켜고 끔(기본 끔) | 글·이미지 | 256K |
| Gemma 4 E2B | 5.1B(계산 2.3B) | 켜고 끔 | 글·이미지·음성 | 128K |
| Ministral 3 3B | 3.8B | 별도 Reasoning 모델 | 글·이미지 | 256K |

- **Gemma 4 E2B의 E:** effective의 약자로, 전체 5.1B 가운데 토큰 하나를 계산하는 본체가 약 2B(2.3B)라는 뜻입니다. 나머지는 층마다 붙은 임베딩 테이블과 이미지·음성 인코더입니다.

### 2-5. 주요 오픈 웨이트 모델 계열

계열별 대표 모델과 라이선스를 알고 있으면 후보를 빨리 좁힙니다. 그룹마다 가장 많이 알려진 계열 세 개 안팎만 골랐습니다.

- **최소 GPU 메모리:** BF16 가중치에 생성 여유 20%를 더한 값입니다(MoE도 전체 파라미터 기준). 괄호는 24GB·32GB GPU 한 장에 들어가는지입니다(둘 다, 32GB만, 4비트면 24GB, GPU 여러 장).

**Qwen(알리바바, 중국)**

| 세대 | 공개 | 오픈 웨이트 크기 | 추론 모드 | 라이선스 | 최소 GPU 메모리(BF16 / 4비트) |
|---|---|---|---|---|---|
| Qwen3 | 2025년 4월 | dense 0.6B~32B, MoE 30B-A3B·235B-A22B | 켜고 끔 | Apache-2.0 | Qwen3-8B: BF16 18.3GiB / 4비트 4.6GiB(둘 다) |
| Qwen3.5(이 교안) | 2026년 2월 | dense 0.8B~27B, MoE 35B-A3B~397B-A17B(모두 이미지 입력) | 켜고 끔 | Apache-2.0 | Qwen3.5-4B: BF16 10.4GiB / 4비트 2.6GiB(둘 다) |
| Qwen3.8 | 2026년 8월 | dense 27B, MoE 2.4T-A95B 등 | 추론 강도 단계(xhigh·medium·low) | 27B는 Apache-2.0, 큰 모델은 자체 라이선스 | Qwen3.8-27B: BF16 62GiB / 4비트 15.5GiB(4비트면 24GB) |


**한국 계열**

| 개발 주체 | 대표 모델(전체 / 활성) | 라이선스 | 최소 GPU 메모리(BF16 / 4비트) |
|---|---|---|---|
| LG AI연구원 | K-EXAONE 2.0(MoE 750B / 37B)<br>EXAONE 4.0 1.2B | K-EXAONE 2.0은 Apache-2.0<br>EXAONE 4.x는 비상업 | K-EXAONE 2.0: BF16 약 1,680GiB / 4비트 약 420GiB(GPU 여러 장)<br>EXAONE 4.0 1.2B: BF16 2.9GiB / 4비트 0.7GiB(둘 다) |
| 업스테이지 | Solar Open 100B(MoE 103B / 12B) | Upstage Solar License(상업 가능, "Built with Solar" 표시) | Solar Open 100B: BF16 약 230GiB / 4비트 57GiB(GPU 여러 장) |
| SK텔레콤 | A.X K2(MoE 692B / 33B)<br>A.X 4.0 Light 7B | Apache-2.0 | A.X K2: BF16 약 1,550GiB / 4비트 약 390GiB(GPU 여러 장)<br>A.X 4.0 Light 7B: BF16 16.2GiB / 4비트 4.1GiB(둘 다) |

- **그 밖의 한국 계열:** 네이버클라우드(HyperCLOVA X SEED), 카카오(Kanana), KT(Mi:dm)도 오픈 웨이트 모델을 공개합니다.

**미국·유럽 주요 계열**

| 계열(개발 주체, 나라) | 대표 모델 | 크기(전체 / 활성) | 라이선스 | 특징 | 최소 GPU 메모리(BF16 / 4비트) |
|---|---|---|---|---|---|
| gpt-oss(OpenAI, 미국) | gpt-oss-20b | 21B / 3.6B | Apache-2.0 | 처음부터 4비트로 배포 | BF16 47GiB / 4비트 11.7GiB(4비트면 24GB) |
| Gemma(Google, 미국) | Gemma 4 31B | 31B(dense) | Apache-2.0 | 이미지 입력, 작은 크기도 있음 | BF16 70GiB / 4비트 17.5GiB(4비트면 24GB) |
| Mistral(Mistral AI, 프랑스) | Mistral Small 4 | 119B / 6.5B | Apache-2.0 | 유럽 대표 | BF16 약 270GiB / 4비트 67GiB(GPU 여러 장) |

중국의 DeepSeek·Kimi·MiniMax 같은 대표 모델은 4비트로도 GPU 한 장을 넘는 초대형이라 2-7에서 따로 봅니다.

**코딩에 강한 모델(4비트로 GPU 한 장에 들어가는 것)**

| 모델 | 크기 | 라이선스 | SWE-bench Verified | Ollama 태그(파일 크기) | 최소 GPU 메모리(BF16 / 4비트) |
|---|---|---|---|---|---|
| Qwen3.6-27B | dense 27B | Apache-2.0 | 77.2 | `qwen3.6:27b`(18GB) | BF16 62GiB / 4비트 15.5GiB(4비트면 24GB) |
| gpt-oss-20b | MoE 21B / 3.6B | Apache-2.0 | 60.7 | `gpt-oss:20b`(14GB) | BF16 47GiB / 4비트 11.7GiB(4비트면 24GB) |
| Qwen3-Coder-30B-A3B | MoE 30.5B / 3.3B | Apache-2.0 | 51.6 | `qwen3-coder:30b`(19GB) | BF16 68GiB / 4비트 17.1GiB(4비트면 24GB) |

- **SWE-bench Verified:** 실제 GitHub 저장소의 이슈를 모델이 코드를 고쳐 해결하는 비율(%)입니다. 개발사마다 자기 도구로 측정한 값이라 참고로만 봅니다.
- **같은 계열의 API 모델:** Qwen3.8-Max, Qwen3.5-Plus, Codestral 25.x는 가중치를 공개하지 않은 API 모델입니다. Hugging Face에 가중치 파일이 있는지로 구분합니다.

### 2-6. 유명하고 최신인 모델: 크기와 용도

파라미터 수가 모델 파일 크기와 필요한 GPU 메모리(VRAM)를 정하므로, 후보는 크기 구간부터 좁힙니다. 많이 알려진 최신 모델을 크기 구간과 용도로 나눴습니다. 괄호는 라이선스입니다(NC는 비상업, 자체는 회사 고유 라이선스). 줄 이름의 최소 GPU 메모리는 2-5와 같은 계산입니다.

| 크기(최소 GPU 메모리) | 일반 대화·추론 | 코딩·에이전트 | 한국어 | 이미지 입력 |
|---|---|---|---|---|
| **4B 이하**(BF16 약 10GiB 이하: 노트북·CPU·기기 안) | Qwen3.5-4B(Apache, 이 교안) | - | EXAONE 4.0 1.2B(NC) | Gemma 4 E2B·E4B(Apache) |
| **4B~15B**(BF16 9~33GiB: GPU 한 장, 14B급은 8비트) | Qwen3.5-9B(Apache)<br>Qwen3-14B(Apache) | Ministral 3 8B·14B(Apache) | A.X 4.0 Light 7B(Apache) | Gemma 4 12B(Apache) |
| **15B~40B**(4비트 8~22GiB: 24GB·32GB 한 장) | Qwen3.8-27B(Apache)<br>gpt-oss-20b(Apache) | Qwen3.6-27B(Apache)<br>Devstral Small 2 24B(Apache) | EXAONE 4.5 33B(NC) | Gemma 4 31B(Apache) |
| **40B~150B**(4비트 약 19~70GiB: 80GB GPU 한 장) | Qwen3.5-122B-A10B(Apache)<br>gpt-oss-120b(Apache) | Mistral Small 4(Apache) | Solar Open 100B(Solar) | Qwen3.5-122B-A10B(Apache) |

- **크기 경계:** 크기는 전체 파라미터 기준입니다. Qwen3.5-4B(전체 4.66B)와 Gemma 4 E2B·E4B(5.1B·8.0B)는 이름의 크기대로 4B 이하 줄에 넣었습니다.
- **이름과 실제 크기:** 이름의 크기는 어림값입니다. Qwen3-14B는 실제 14.8B라 32GB의 BF16 한도(약 14B)를 넘습니다.
- **150B 초과:** 4비트로도 GPU 한 장에 들어가지 않습니다(2-7).
- **쓰는 법:** 장비로 줄을, 업무로 칸을 정한 뒤 3절의 순서로 고릅니다.

### 2-7. 4비트로도 GPU 한 장에 들어가지 않는 초대형 모델

4비트로 줄여도 가중치가 80GB GPU 한 장을 넘는 모델(전체 약 150B 이상)입니다. 성능 점수가 가장 높은 오픈 웨이트 모델은 대부분 이 크기의 MoE입니다.

| 모델 | 개발 주체(나라) | 전체 / 활성 | 4비트 가중치 어림(전체 x 0.5바이트) | 라이선스 | 특징 |
|---|---|---|---|---|---|
| Kimi K3 | 문샷 AI(중국) | 2.8T / 104B | 약 1,300GiB | 자체(API 사업 매출 조건) | 가장 큰 축의 공개 모델, 긴 코딩·에이전트 작업 |
| DeepSeek V4 Pro | DeepSeek(중국) | 1.6T / 49B | 약 745GiB | MIT | 코딩 에이전트 점수 상위 |
| MiMo-V2.6-Pro | 샤오미(중국) | 1.02T / 42B | 약 475GiB | MIT | 종합 지수 최상위권 |
| GLM-5.3 | Z.ai(중국) | 753B / 40B | 약 350GiB | 자체 | 코딩 에이전트 점수 상위 |
| DeepSeek V4.1 Flash | DeepSeek(중국) | 552B / 16B | 약 257GiB | MIT | 활성이 작아 빠르고 싸서 API 사용량이 많음 |
| MiniMax-M3 | MiniMax(중국) | 428B / 23B | 약 200GiB | 자체(표시·매출 조건) | 싸고 빠른 코딩 에이전트 |
| Qwen3.5-397B-A17B | 알리바바(중국) | 397B / 17B | 약 185GiB | Apache-2.0 | 실습 모델과 같은 세대의 가장 큰 모델 |

- **실행:** GPU 여러 장의 서버나 API로 씁니다. Ollama의 `:cloud` 태그(예: `minimax-m3:cloud`)는 내 GPU가 아니라 Ollama 서버에서 실행되므로 로컬 실행이 아닙니다.
- **MoE:** 모두 MoE라 토큰 하나의 계산은 활성 파라미터만큼이지만, GPU 메모리에는 전체를 올려야 합니다(2-3).

## 3. 모델 고르기

모델은 요구 조건, 라이선스, 성능, 크기와 VRAM으로 후보를 줄인 뒤 실행해서 답을 확인하고 고릅니다. 후보 여럿을 비교할 때는 같은 실행 코드에 모델 이름만 바꿔 같은 질문을 넣습니다.

<img src="images/교안/24_model_choice_flow.png" width="1050" alt="제목은 '모델 고르기: 조건으로 후보를 줄이고 실행해서 비교'입니다. 1 요구 조건(한국어 · 상업 서비스 · 개인정보), 2 라이선스(상업 사용 허용 여부), 3 크기와 VRAM(BF16은 10억 개당 약 1.86GiB), 4 실행해서 비교(같은 질문으로 답 확인). 아래 깔때기에서 후보 카드가 여섯 장, 세 장, 초록 테두리 한 장으로 줄고 개발자가 마지막 한 장을 가리킵니다. 결론: 조건에 맞는 가장 작은 모델부터.">

### 3-1. 이름 읽기

모델 이름에는 계열, 크기, 학습 단계, 저장 형식이 들어 있습니다.

- **`Qwen3.5-4B`:** 계열 이름과 세대(`Qwen3.5`), 파라미터 수(`4B`, B는 10억)입니다. 이름의 크기는 어림값이라 실제 수(4.66B)는 모델 카드에서 확인합니다.
- **`-Instruct`, `-it`:** base 모델(사전학습만 한 모델로, 앞 글을 이어 쓰기만 합니다)에 지시를 따르도록 추가 학습한 버전입니다. 질문에 답하는 챗봇에는 이 버전을 씁니다. base 모델에 지시 학습을 더하는 방법은 튜닝 단원에서 직접 해 봅니다.
- **꼬리가 없는 이름:** Qwen3.5는 대화용을 꼬리 없이(`Qwen3.5-4B`), base 모델을 `-Base`로 올립니다. 모델 카드에서 확인합니다.
- **`35B-A3B`, `26B-A4B`:** MoE 모델의 전체 파라미터와 활성 파라미터입니다(2-3).
- **`E2B`, `E4B`:** Gemma 4 작은 모델의 이름입니다(E의 뜻은 2-4).
- **`-Thinking`, `-Reasoning`, `-Think`:** 추론 모드 전용으로 학습한 버전입니다.
- **`-2507`, `-2512`:** 공개한 연월(2025년 7월, 2025년 12월)입니다.
- **`-FP8`, `-AWQ`, `-GPTQ`, `-GGUF`:** 원본을 다른 정밀도나 양자화 형식으로 바꿔 올린 저장소입니다. FP8은 8비트 실수 정밀도, AWQ·GPTQ는 양자화 방법 이름, GGUF는 파일 형식(1-1)입니다(교안 02 6-1·6-3).

### 3-2. 라이선스는 모델마다

같은 회사·같은 계열이라도 크기와 세대마다 라이선스가 달라서(예: EXAONE 4.0은 비상업, K-EXAONE 2.0은 Apache-2.0), 상업 서비스에 쓸 수 있는지는 저장소마다 확인합니다.

| 라이선스 | 해당 모델 예 | 상업 사용 | 지켜야 할 조건 |
|---|---|---|---|
| Apache-2.0 | Qwen3·Qwen3.5, Gemma 4, gpt-oss, Mistral Small 4, K-EXAONE 2.0, A.X | 가능 | 라이선스 사본 포함, 저작권·NOTICE 고지 유지, 바꾼 파일 표시 |
| MIT | DeepSeek, Phi-4, Mi:dm 2.0 | 가능 | 저작권·라이선스 고지 유지 |
| 규모·표시 조건이 붙은 자체 라이선스 | Llama 4(월간 활성 사용자 7억 명 초과 시 별도 허가), Kimi K3(API 사업 매출 2천만 달러 초과 시 별도 계약), HyperCLOVA X SEED(월간 활성 사용자 1천만 명 초과 또는 NAVER와 경쟁하는 서비스는 별도 계약), Solar Open(파생 모델 이름 앞에 Solar) | 조건부 | 사용자 수·매출 기준, "Built with ..." 표시, 파생 모델 이름 규칙 |
| 비상업 라이선스 | Qwen2.5-3B(Qwen Research), EXAONE 4.x(EXAONE NC) | 불가 | 연구·비상업 용도만 |
| 게이트가 있는 약관 | Llama 3.1·3.2·4, Gemma 3(Gemma 약관) | 조건부 | 약관 동의·승인 뒤 내려받기, 금지 용도 정책 준수 |

**회사 서비스에 넣기 전 체크리스트**
1. 상업 사용이 허용되는가?
2. 월간 활성 사용자 수나 매출 같은 규모 조건이 붙는가?
3. 우리 용도가 금지 용도 정책에 걸리지 않는가?
4. 튜닝한 모델을 배포할 때 지켜야 할 고지·이름 규칙이 있는가?

- **게이트(gated) 모델:** Llama·Gemma 3처럼 저장소에서 약관에 동의하고 승인을 받아야 내려받을 수 있는 모델입니다. 로그인한 뒤 모델 페이지의 약관 동의 양식을 제출하고, 승인 뒤 액세스 토큰으로 내려받습니다([Hugging Face 게이트 모델 안내](https://huggingface.co/docs/hub/models-gated)).

<img src="images/교안/cap_hf_05_gated.png" width="1050" alt="Hugging Face의 meta-llama/Llama-3.1-8B-Instruct 모델 페이지 화면입니다. 빨간 사각형은 모델을 내려받기 전에 연락처를 공유하고 약관에 동의하라는 게이트 양식을 표시합니다.">

- 누를 곳: Hugging Face에 로그인 -> 모델 페이지 위쪽의 약관 동의 패널
- 주소: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct (로그인한 화면)

### 3-3. 벤치마크와 리더보드

라이선스와 크기를 통과한 후보끼리는 성능으로 줄입니다. 성능은 직접 실행하기 전에 공개된 벤치마크 점수로 먼저 어림합니다.

- **벤치마크(benchmark):** 정해 둔 질문 묶음과 채점 방법입니다. 같은 질문으로 여러 모델을 채점해 점수를 비교합니다.
- **리더보드(leaderboard):** 여러 모델의 벤치마크 점수나 사람 투표 결과를 한 테이블로 모아 순위를 매긴 사이트입니다.

**자주 나오는 벤치마크**

| 벤치마크 | 평가하는 능력 | 형식 |
|---|---|---|
| MMLU-Pro | 여러 분야 지식과 추론 | 10지선다 |
| GPQA Diamond | 대학원 수준 과학 | 4지선다 |
| AIME | 고등학생 수학 경시 문제 | 정답 숫자 |
| LiveCodeBench | 코드 작성(공개 뒤에 나온 새 문제) | 코드 실행 채점 |
| KMMLU | 한국어 전문 지식 | 4지선다 |

**리더보드 사이트**

| 사이트 | 순위를 매기는 방법 |
|---|---|
| LMArena(https://arena.ai/leaderboard) | 사람이 두 모델의 답을 보고 더 나은 쪽에 투표한 결과 |
| Artificial Analysis(https://artificialanalysis.ai/models/open-source) | 여러 벤치마크를 같은 조건으로 직접 돌려 계산한 종합 지수(Intelligence Index) |
| Hugging Face 벤치마크 리더보드 | 벤치마크 데이터셋 페이지마다 모델 점수를 모은 순위. 모델 카드의 Evaluation results에도 등록된 점수가 나옴 |

<img src="images/교안/cap_bench_01_lmarena.png" width="1050" alt="LMArena 텍스트 리더보드 화면입니다. 왼쪽 빨간 사각형은 License Type 필터(All, Proprietary, Open Source)에서 Open Source를 고른 상태를 표시합니다. 오른쪽 빨간 사각형은 1~3위 모델 칸으로, 모델 이름 아래에 개발사와 라이선스(Moonshot · Kimi K3 license, Z.ai · MIT, Xiaomi · MIT)가 붙어 있습니다.">

- 누를 곳: arena.ai -> Leaderboard -> Text -> 왼쪽 License Type에서 Open Source
- 주소: https://arena.ai/leaderboard/chat/text?license=open-source (LMArena는 arena.ai 주소로 옮겨 갔습니다)
- **읽는 법:** 순위의 "오픈" 표시만 보고 상업 서비스에 쓰지 말고, 모델 이름 아래 줄의 라이선스를 3-2의 체크리스트로 확인합니다. Score 옆의 ± 값은 점수의 불확실한 폭이라, 폭이 겹치는 모델끼리는 순위 차이를 크게 보지 않습니다.

<img src="images/교안/cap_bench_02_aa.png" width="1050" alt="Artificial Analysis의 Comparison of Open Source Models 화면입니다. 위쪽에 크기 구간 탭(All, Large, Medium, Small 4B~40B, Tiny)이 있고, 아래에 막대 그래프 세 개가 있습니다. 왼쪽은 공개 정도를 매긴 Openness(1위 Nemotron 3.5 Super VL 83, DeepSeek V4.1 Flash 44), 가운데 빨간 사각형은 종합 지수 Intelligence(1위 MiMo-V2.6-Pro 46), 오른쪽은 전체 파라미터 수입니다.">

- 누를 곳: artificialanalysis.ai -> Models -> Open Source Models -> 위쪽 탭에서 크기 구간
- 주소: https://artificialanalysis.ai/models/open-source
- **Openness Index:** 모델이 얼마나 공개되어 있는지를 0~100으로 매긴 점수입니다(성능 점수가 아닙니다).
- **매기는 방법:** 사용 가능성(6점: 가중치를 내려받을 수 있는지, 라이선스가 상업 사용을 얼마나 자유롭게 허용하는지)과 투명성(12점: 학습 데이터와 그 라이선스, 학습 방법, 학습 코드의 공개)을 더한 18점을 0~100으로 바꿉니다.
- **읽는 법:** 가중치만 공개하면 낮고 데이터·학습 코드까지 공개하면 높습니다(Nemotron 3.5 Super VL 83, DeepSeek V4.1 Flash 44). 성능(Intelligence)과 함께 봅니다.

<img src="images/교안/cap_site_hf_bench.png" width="1050" alt="Hugging Face의 GPQA Diamond 벤치마크 데이터셋 페이지에 있는 Leaderboard 화면입니다. 왼쪽 위 빨간 사각형은 Parameters Size 필터(All, <1B, <3B, <6B, <12B, <32B, <128B, <500B)에서 <12B를 고른 상태이고, 오른쪽 빨간 사각형은 SCORE 열입니다. 점수 옆 별표(*)는 개발사가 아닌 사람이 올린 점수입니다.">

- 누를 곳: 벤치마크 데이터셋 페이지(예: Idavidrein/gpqa) -> Leaderboard -> Parameters Size
- 주소: https://huggingface.co/datasets/Idavidrein/gpqa
- **예전 리더보드:** Hugging Face Open LLM Leaderboard는 보관(Archived) 상태라, 지금은 벤치마크 데이터셋마다 리더보드를 둡니다.
- **별표(*) 점수:** 개발사가 아닌 사람이 PR(수정 요청)로 올린 점수입니다. 누구나 올릴 수 있고 개발사가 확인하지 않았으므로, 개발사가 모델 카드 본문에 직접 올린 같은 벤치마크 점수와 비교한 뒤 씁니다.

**크기 구간별 성능**

Intelligence Index에 묶인 평가는 에이전트 작업, 터미널 작업, 과학 코딩, Humanity's Last Exam 등 어려운 평가 10개입니다. 아래 그림은 사이트의 크기 탭(4B 이하, 4B~40B, 40B~150B, 150B 초과)에 있는 주요 모델의 크기와 점수, 실습 모델의 위치를 보여 줍니다(빈 점과 "추정"은 사이트가 추정치로 표시한 값).

<img src="images/교안/28_size_vs_score.png" width="1050" alt="제목은 '오픈 웨이트 LLM의 크기와 성능(Artificial Analysis Intelligence Index)'입니다. 가로축은 전체 파라미터 수(로그 눈금, 1B~1T), 세로축은 Intelligence Index(0~50)입니다. 점선으로 4B 이하, 4B~40B, 40B~150B, 150B 초과 구간을 나누고, 진한 초록은 24GB(약 10B까지), 연한 초록은 32GB(약 14B까지)에 BF16으로 들어가는 크기입니다. 색은 나라, 원은 dense, 마름모는 MoE, 빈 점은 추정치입니다. 150B 초과의 MiMo-V2.6-Pro·DeepSeek V4.1 Flash·GLM-5.3이 45 안팎으로 가장 높고, 4B~40B에서는 Qwen3.8 27B가 33.7, K2 Horizon 7B가 20.6입니다. 화살표로 표시한 실습 모델 Qwen3.5-4B는 빈 점(추정치) 13.1로, 이전 세대 Qwen3-8B(5.2)보다 높습니다. 한국 모델 Solar Open2 250B, A.X-K2, K-EXAONE 2.0은 20 안팎입니다.">

- **크기와 점수:** 같은 구간 안에서도 세대에 따라 몇 배 차이가 납니다. Qwen3.8 27B(33.7)는 150B를 넘는 모델 여럿보다 높습니다.
- **실습 모델의 위치:** Qwen3.5-4B는 13.1(추정)로, 크기가 두 배인 이전 세대 Qwen3-8B(5.2)보다 높고 같은 구간의 K2 Horizon 7B(20.6)보다는 낮습니다.

**점수를 읽을 때 주의할 점**
- **설정이 다르면 점수도 다릅니다:** 추론 모드를 켰는지, 몇 번 생성해 평균했는지에 따라 점수가 바뀌므로 같은 조건의 점수끼리만 비교합니다.
- **결정은 우리 질문으로:** 벤치마크는 후보를 줄이는 데 쓰고, 마지막 결정은 우리 업무 질문을 후보마다 실행해 내립니다(4·5절의 실행 방법).

### 3-4. Hub에서 후보 찾고 테이블 만들기

Hugging Face Hub의 모델 검색 화면에 계열 이름을 넣으면 그 계열의 저장소가 모입니다. 왼쪽 필터의 하는 일(task)과 파라미터 수로 더 좁힐 수 있습니다.

<img src="images/교안/cap_hf_04_search.png" width="1050" alt="Hugging Face 모델 검색 화면입니다. 위쪽 빨간 사각형은 검색창에 Qwen3.5를 넣어 나온 모델 수이고, 아래 빨간 사각형은 개발사 공식 저장소 Qwen/Qwen3.5-4B 카드(Image-Text-to-Text, 5B)입니다. 그 위에는 다른 사용자가 고쳐 올린 Qwen3.5 저장소들이 함께 보입니다.">

- 누를 곳: huggingface.co -> Models -> 검색창에 Qwen3.5
- 주소: https://huggingface.co/models?search=Qwen3.5
- **저장소 주인 확인:** 개발사 공식 저장소(`Qwen/`)와 다른 사람이 고쳐 올린 저장소가 섞여 나오므로, 이름 앞의 주인을 보고 원본을 고릅니다.

후보를 누르면 모델 카드가 열립니다. 모델 카드 위쪽의 태그에서 하는 일과 라이선스를, 오른쪽에서 모델 크기와 저장 정밀도를 확인합니다.

<img src="images/교안/cap_hf_01_model_card.png" width="1050" alt="Hugging Face의 Qwen/Qwen3.5-4B 모델 카드 화면입니다. 빨간 사각형은 하는 일 태그 Image-Text-to-Text, 라이선스 태그 apache-2.0, 그리고 오른쪽 Safetensors 칸의 Model size 5B params · Tensor type BF16 · F32를 표시합니다.">

- 누를 곳: 검색 결과에서 Qwen/Qwen3.5-4B -> Model card 탭
- 주소: https://huggingface.co/Qwen/Qwen3.5-4B
- **화면의 5B:** 이미지 인코더까지 센 전체 4.66B를 반올림한 값입니다. Tensor type의 F32는 일부 텐서가 32비트로 저장되어 있다는 뜻입니다.

모델 카드를 하나씩 열어 보는 대신, 후보의 라이선스·게이트·크기를 코드로 모아 한 테이블에서 비교합니다.

- **safetensors:** Hugging Face가 가중치를 저장하는 파일 형식입니다(`model.safetensors`, 큰 모델은 여러 파일로 나눔).
- **`HfApi().model_info(저장소 이름, expand=[...])`:** Hugging Face Hub에서 모델 저장소 정보를 조회합니다. 가중치는 내려받지 않습니다. `expand`로 받을 항목을 고릅니다.
  - `cardData`: 모델 카드 머리말(라이선스 등)
  - `safetensors`: 가중치 파일에 담긴 정밀도별 파라미터 수
  - `gated`: 게이트 여부(`False`면 바로 받기, `auto`면 약관 동의 즉시 승인, `manual`이면 사람의 승인 필요)
  - `pipeline_tag`: 모델이 하는 일. 글만 받는 생성 모델은 `text-generation`, 이미지도 받는 모델은 `image-text-to-text`·`any-to-any`입니다

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# 실습 모델 하나의 저장소 정보를 조회합니다. 받은 네 항목을 하나씩 원형 그대로 봅니다
hub_info = api.model_info("Qwen/Qwen3.5-4B", expand=["cardData", "safetensors", "gated", "pipeline_tag"])

# 1) 모델 카드 머리말
display(hub_info.card_data)

In [ ]:
# 2) 가중치 파일의 정밀도별 파라미터 수(parameters)와 전체 파라미터 수(total)
display(hub_info.safetensors)

In [ ]:
# 3) 게이트 여부, 4) Hub가 분류한 모델이 하는 일
print("게이트:", hub_info.gated)
print("하는 일:", hub_info.pipeline_tag)

- **모델 카드 머리말:** `license: apache-2.0`가 화면의 라이선스 태그이고, `base_model`은 이 모델의 바탕이 된 모델(`Qwen3.5-4B-Base`)입니다.
- **safetensors:** 저장 정밀도별 파라미터 수(BF16 4,659,861,248개, F32 3,840개)와 그 합 `total`(4,659,865,088개, 화면의 5B)입니다.
- **게이트·하는 일:** `False`라 약관 동의 없이 바로 받고, `image-text-to-text`(글과 이미지를 받아 글을 생성)입니다.

같은 조회를 후보마다 반복해 한 테이블로 모읍니다. 한 행을 만드는 일은 여기와 6절 함께 따라하기에서 두 번 쓰므로 함수 `candidate_row`로 만듭니다. `BF16 가중치 GiB` 열은 맨 위 단위 테이블의 계산(파라미터 수 x 2바이트)입니다.

In [ ]:
def candidate_row(repo_id):
    """모델 저장소 정보를 조회해 후보 테이블의 한 행(dict)으로 돌려줍니다."""
    # 가중치는 받지 않고 모델 카드, safetensors 요약, 게이트 정보만 조회합니다
    info = api.model_info(repo_id, expand=["cardData", "safetensors", "gated", "pipeline_tag"])

    # 파라미터 수는 safetensors 요약의 total, 저장 정밀도는 parameters의 키(예: BF16)입니다
    total_params = info.safetensors.total

    # 16비트(2바이트)로 불러올 때 가중치 메모리를 GiB(1024의 세제곱 바이트)로 계산합니다
    bf16_gib = total_params * 2 / 1024**3
    return {
        "모델": repo_id,
        "라이선스": info.card_data.license,
        "라이선스 이름": info.card_data.license_name,
        "게이트": info.gated,
        "하는 일": info.pipeline_tag,
        "파라미터 수": total_params,
        "저장 정밀도": list(info.safetensors.parameters),
        "BF16 가중치 GiB": round(bf16_gib, 2),
    }

In [ ]:
# 비교할 후보 모델의 저장소 이름
candidate_ids = [
    "Qwen/Qwen3.5-4B",
    "Qwen/Qwen3-14B",
    "Qwen/Qwen3.5-9B",
    "google/gemma-4-E4B-it",
    "meta-llama/Llama-3.1-8B-Instruct",
    "LGAI-EXAONE/EXAONE-4.0-32B",
    "skt/A.X-4.0-Light",
    "K-intelligence/Midm-2.0-Base-Instruct",
]

# 후보마다 candidate_row로 한 행씩 만들어 테이블로 모읍니다
candidate_rows = []
for repo_id in candidate_ids:
    candidate_rows.append(candidate_row(repo_id))
candidates = pd.DataFrame(candidate_rows)
display(candidates)

테이블을 체크리스트 순서대로 읽습니다.

- **라이선스:** `other`는 Hub의 표준 목록에 없는 자체 라이선스라는 뜻입니다. 라이선스 이름 열의 `exaone`(EXAONE 4.0)은 비상업 라이선스입니다. Llama는 라이선스 열 자체가 `llama3.1` 약관입니다.
- **게이트:** Llama 3.1은 `manual`이라 승인을 받아야 내려받을 수 있습니다(3-2의 화면).
- **하는 일:** Qwen3.5(`image-text-to-text`)는 `AutoModelForCausalLM`으로 글 생성 부분만 불러올 수 있습니다(3-5). Qwen3-VL처럼 이 방식으로 불러오지 못하는 모델도 있고, Gemma 4 E4B(`any-to-any`)는 이미지·음성 인코더까지 함께 올라옵니다.
- **크기:** 생성할 때는 BF16 가중치에 약 20%가 더 듭니다(교안 02 2-4). 그래서 가중치가 24GB GPU는 약 20GiB, 32GB GPU는 약 26.7GiB 이하여야 합니다.
  - **둘 다 들어감:** Qwen3.5-4B(8.68GiB), A.X 4.0 Light(13.52GiB), Gemma 4 E4B(14.89GiB), Llama 3.1 8B(14.96GiB), Qwen3.5-9B(17.98GiB)
  - **32GB에만 들어감:** Midm 2.0 Base(21.51GiB)는 20%를 더하면 약 25.8GiB라 24GB를 넘습니다.
  - **둘 다 넘침:** Qwen3-14B(27.51GiB)는 가중치는 32GB에 들어가지만 20%를 더하면 약 33GiB입니다. EXAONE 4.0 32B(59.61GiB)는 라이선스와 크기 두 조건에서 떨어집니다.

**이 교안의 선택:** 상업 사용이 가능하고(Apache-2.0) 게이트가 없으며 `AutoModelForCausalLM`으로 글 생성 부분을 불러올 수 있는 후보 가운데 가장 작은 `Qwen/Qwen3.5-4B`를 고릅니다. 2026년에 나온 세대라 같은 크기에서 점수가 높고(3-3), Ollama 공식 태그(`qwen3.5:4b-q4_K_M`)가 있으며, 추론 모드를 켜고 끌 수 있습니다. 같은 조건을 통과한 A.X 4.0 Light(Apache-2.0)와 Midm 2.0 Base(MIT, 32GB GPU에서만)는 한국 모델이라, 한국어 답의 품질이 더 중요하면 같은 질문 묶음으로 비교할 다음 후보입니다.

### 3-5. 실습 모델 Qwen3.5-4B와 설정 파일

3-4에서 고른 Qwen3.5-4B를 모델 카드로 정리합니다.

| 항목 | 내용 |
|---|---|
| 라이선스 | Apache-2.0(상업 사용 가능, 게이트 없음) |
| 입력 | 글·이미지(`image-text-to-text`). 이미지 입력은 Ollama(4-6)로, Transformers(5절)는 글 생성 부분만 씁니다 |
| 크기 | 전체 4,659,865,088개(BF16 8.68GiB, 이미지 인코더 포함). 글 생성 부분 4,205,751,296개(BF16 7.83GiB) |
| 구조 | 32층 = 8 x (Gated DeltaNet 3층 -> 어텐션 1층), 은닉 크기 2,560, GQA |
| 컨텍스트 | 262,144토큰(모델 카드: 약 100만 토큰까지 확장 가능) |

- **Gated DeltaNet(선형 어텐션):** 앞 토큰을 모두 다시 보는 대신, 입력 길이와 상관없는 고정 크기의 상태에 정보를 누적해 갱신하는 층입니다. 입력이 길어도 이 층의 메모리는 늘지 않아, KV 캐시는 어텐션 8층에만 쌓입니다(5절).
- **GQA(grouped-query attention):** 질문(Q) 헤드 여러 개가 키·값(K·V) 헤드 하나를 함께 쓰는 어텐션입니다. 저장해 두는 K·V가 줄어 생성할 때 메모리가 적게 듭니다.

**벤치마크(모델 카드·기술 보고서, 추론 모드 켬)**

| 벤치마크 | 평가하는 능력 | Qwen3.5-4B | Qwen3-8B(이전 세대) |
|---|---|---|---|
| GPQA-Diamond | 대학원 수준 과학 | 76.2 | 62.0 |
| IFEval | 지시 따르기 | 89.8 | 85.0 |
| MMMLU | 여러 언어 지식 | 76.1 | 74.4 |
| INCLUDE | 여러 나라 언어의 지역 지식 | 71.0 | 67.8 |
| MMLU-Pro | 여러 분야 지식과 추론 | 79.1 | - |
| LiveCodeBench v6 | 코드 작성 | 55.8 | - |

- **세대 차이:** 크기는 Qwen3-8B의 절반인데 같은 벤치마크 점수가 더 높습니다.
- **한국어:** 모델 카드에 한국어 전용 점수(KMMLU)는 없습니다. MMMLU·INCLUDE 같은 다국어 점수로 어림하고, 한국어 업무는 직접 질문해 확인합니다.

내려받기 전에 모델의 구조와 한 번에 넣을 수 있는 길이를 설정 파일(`config.json`)에서 확인합니다.

<img src="images/교안/cap_hf_03_config.png" width="1050" alt="Hugging Face에서 Qwen/Qwen3.5-4B의 config.json을 연 화면입니다. text_config 안에서 빨간 사각형은 layer_types의 첫 묶음(linear_attention 3개 뒤에 full_attention 1개)과 num_hidden_layers 32 줄을 표시합니다. 같은 묶음이 아래로 8번 반복됩니다.">

- 누를 곳: 모델 페이지 -> Files and versions 탭 -> config.json
- 주소: https://huggingface.co/Qwen/Qwen3.5-4B/blob/main/config.json
- **`architectures`의 `Qwen3_5ForConditionalGeneration`:** 이미지까지 받는 전체 모델의 클래스입니다. 5-1의 `AutoModelForCausalLM`은 이 가운데 글 생성 부분(`Qwen3_5ForCausalLM`)만 불러옵니다.

2-3과 같은 `AutoConfig.from_pretrained`로 읽습니다. 2-3의 MoE 모델처럼 이미지도 받는 모델이라, 글 생성 부분의 설정은 `text_config` 안에 있습니다.

In [ ]:
# 3-4에서 고른 모델. 이 교안의 Transformers 실습은 모두 이 이름을 씁니다
MODEL_NAME = "Qwen/Qwen3.5-4B"

# 가중치는 받지 않고 설정 파일(config.json)만 읽습니다
config = AutoConfig.from_pretrained(MODEL_NAME)
display(config)

In [ ]:
# 글 생성 부분의 설정
text_config = config.text_config

# 모델을 고를 때 보는 설정 값만 골라 테이블로 봅니다. 어텐션 층 수는 layer_types에서 full_attention을 셉니다
config_rows = [
    {"항목": "층 수", "설정 이름": "num_hidden_layers", "값": text_config.num_hidden_layers},
    {"항목": "어텐션 층 수", "설정 이름": "layer_types", "값": text_config.layer_types.count("full_attention")},
    {"항목": "은닉 크기", "설정 이름": "hidden_size", "값": text_config.hidden_size},
    {"항목": "어휘 수", "설정 이름": "vocab_size", "값": text_config.vocab_size},
    {"항목": "컨텍스트 길이", "설정 이름": "max_position_embeddings", "값": text_config.max_position_embeddings},
    {"항목": "저장 정밀도", "설정 이름": "dtype", "값": text_config.dtype},
]
display(pd.DataFrame(config_rows))

- **층 수 32, 어텐션 층 수 8:** 나머지 24층은 Gated DeltaNet(`linear_attention`)입니다.
- **은닉 크기 2,560:** 토큰 하나를 나타내는 벡터의 길이입니다.
- **어휘 수 248,320:** 모델이 다음 토큰을 고를 때 점수를 매기는 후보의 개수입니다.
- **컨텍스트 길이 262,144:** 입력과 생성한 답을 합친 최대 토큰 수입니다.

## 4. Ollama로 실행하기

코드를 거의 쓰지 않고 모델을 돌려 보는 가장 빠른 방법이 Ollama라서, 먼저 Ollama로 실행합니다.

- **Ollama:** GGUF 모델 파일(1-1)을 내려받아 명령 한 줄로 실행해 주는 로컬 LLM 실행 도구입니다. 내 컴퓨터(또는 파드)에서 서버(`localhost:11434`)로 돌고, 파이썬 `ollama` 라이브러리는 이 서버의 REST API(주소로 요청을 보내고 응답을 받는 방식)를 부릅니다.
- **Q4_K_M:** Ollama 기본 태그가 쓰는 4비트 양자화 수준입니다. 양자화(quantization)는 가중치를 적은 비트로 줄여 저장하는 것입니다(교안 02).

<img src="images/교안/25_ollama_vs_transformers.png" width="1050" alt="제목은 '로컬 실행의 두 방법: Ollama와 Transformers'입니다. Ollama: GGUF 파일, Ollama 서버, ollama.chat(), 명령 한 줄로 받고 실행, 4비트 Q4_K_M이 기본. Transformers: safetensors 파일, 파이썬 모델 객체, model.generate(), 토큰과 생성 설정을 직접 제어, 튜닝으로 이어짐. 아래: 같은 모델을 두 방법으로 실행.">

Ollama에서 받을 모델은 Ollama 라이브러리 사이트에서 찾습니다. 모델 이름 아래의 배지가 지원 기능(추론 모드 `thinking`, 도구 호출 `tools`, 사진 입력 `vision`)이고, 목록의 태그 이름이 `ollama.pull`에 넣을 이름이고, 이 교안은 양자화 수준까지 적은 `qwen3.5:4b-q4_K_M`를 씁니다(4-1).

<img src="images/교안/cap_ollama_01_library.png" width="1050" alt="Ollama 라이브러리의 qwen3.5 페이지 화면입니다. 빨간 사각형은 기능 배지(vision, tools, thinking)와 qwen3.5:4b 행(3.3GB - 4.0GB, 컨텍스트 256K, 입력 Text·Image)을 표시합니다. 맨 위 qwen3.5:latest 행(6.6GB - 7.6GB)은 크기를 적지 않은 태그입니다.">

- 누를 곳: ollama.com -> Models -> qwen3.5 검색
- 주소: https://ollama.com/library/qwen3.5
- **3.3GB - 4.0GB:** 3.3GB가 이 교안이 쓰는 Q4_K_M 파일(4비트 모델 약 2.65GB와 사진을 읽는 부분인 비전 projector 약 0.68GB의 합)이고, 4.0GB는 맥에서 받아지는 MLX 판입니다.

### 4-1. 모델 받기와 대화

- **`ollama.pull("qwen3.5:4b-q4_K_M")`:** Ollama 서버가 모델 파일을 내려받게 합니다. 맨 위 미리 내려받기 셀이 같은 일을 비동기 클라이언트의 `ollama.AsyncClient().pull`로 Hugging Face 다운로드와 함께 이미 했으므로, 여기서는 다시 받지 않습니다.
- **태그는 크기까지:** `qwen3.5`의 latest는 9b라서, `ollama run qwen3.5`처럼 크기 없이 쓰면 9b를 받습니다.
- **양자화 수준까지:** `qwen3.5:4b`는 리눅스에서는 이 파일과 같지만 맥에서는 다른 형식(MLX)이 받아질 수 있어, 이 교안은 양자화 수준까지 적은 `qwen3.5:4b-q4_K_M`를 씁니다.
- **버전:** `qwen3.5` 태그는 Ollama 0.30.0 이상에서 실행됩니다. 설치 가이드의 설치 명령은 최신 버전을 설치합니다.

`ollama.chat`에는 OpenAI API 단원과 같은 형식의 `messages`(역할과 내용의 목록)를 넘깁니다.

- **`think=False`:** 추론 모드(2-4, 기본 켬)를 끕니다.
- **`options`:** 생성 설정을 담습니다. `temperature: 0`은 늘 가장 확률이 높은 토큰을 골라 같은 질문에 같은 답을 내고(5-4), `num_predict: 200`은 새로 만들 토큰의 최대 개수입니다.

In [ ]:
# 헬프데스크 안내원 역할(system)과 직원 질문(user)
helpdesk_system = "당신은 사내 IT 헬프데스크 안내원입니다. 직원의 질문에 한국어로 짧고 단계별로 답합니다."
helpdesk_messages = [
    {"role": "system", "content": helpdesk_system},
    {"role": "user", "content": "VPN이 자꾸 끊길 때 확인할 점을 알려 주세요."},
]

# 추론 모드를 끄고, 같은 답이 나오도록 temperature 0, 새 토큰은 200개까지만 만듭니다
response = ollama.chat(model="qwen3.5:4b-q4_K_M", messages=helpdesk_messages, think=False, options={"temperature": 0, "num_predict": 200})

# 응답 원형: 답 문장과 함께 토큰 수와 걸린 시간이 들어 있습니다
display(response.model_dump())

- **`message.content`:** 모델이 만든 답입니다. `message.thinking`은 추론 모드를 껐으므로 `None`입니다.
- **`prompt_eval_count`:** 입력 토큰 수입니다. system·user 메시지가 chat template으로 바뀐 뒤의 토큰 수입니다.
- **`eval_count`, `eval_duration`:** 생성한 토큰 수와 생성에 걸린 시간입니다. 시간 단위는 나노초(10억 분의 1초)입니다.
- **`load_duration`:** 모델을 GPU에 올린 시간입니다. 첫 호출은 모델을 올리고 계산을 준비하느라 1분 안팎 걸릴 수 있고, 다음 호출부터는 몇 초면 답합니다. 마지막 요청 뒤 5분이 지나면 Ollama가 모델을 내려 다시 느려집니다.
- **`done_reason`:** 생성을 멈춘 이유입니다. `length`는 `num_predict`에 닿아 멈춘 것이고, `stop`은 모델이 답을 끝맺은 것입니다.

In [ ]:
# 모델이 만든 답만 꺼내 봅니다
print(response.message.content)

응답의 토큰 수와 시간으로 생성 속도를 계산합니다. 속도는 1초에 생성하는 토큰 수(토큰/초)로 나타냅니다.

In [ ]:
# 토큰/초 = 생성한 토큰 수 / 생성에 걸린 시간(초). eval_duration은 나노초라서 1e9를 곱해 초 단위로 맞춥니다
tokens_per_sec = response.eval_count / response.eval_duration * 1e9
print("생성 토큰 수:", response.eval_count, "| 토큰/초:", round(tokens_per_sec, 1))

토큰/초는 GPU·CPU 종류와 그때 장치가 하는 다른 일에 따라 달라지고, 첫 호출은 낮게 나올 수 있습니다. 교안 02에서 양자화 수준이 다른 태그끼리 같은 방법으로 비교합니다.

### 4-2. 모델 정보 보기

받은 파일의 형식·양자화 수준·지원 기능·기본 생성 설정을 Ollama가 가진 모델 정보로 확인합니다.

- **`ollama.show(태그)`:** 모델 파일의 세부 정보(`details`), 지원 기능(`capabilities`), 라이선스 원문(`license`), 기본 생성 설정(`parameters`)을 돌려줍니다.

In [ ]:
# 받은 모델의 세부 정보: 파일 형식, 계열, 파라미터 수, 양자화 수준
show_info = ollama.show("qwen3.5:4b-q4_K_M")
display(show_info.details)

- **`parameter_size`:** Ollama가 표시하는 파라미터 수입니다. 3-4 테이블의 파라미터 수와 비교해 봅니다.
- **`quantization_level='Q4_K_M'`:** 태그에 적은 4비트 Q4_K_M 파일입니다. 교안 02에서 다른 양자화 태그와 비교합니다.

In [ ]:
# 지원 기능, 라이선스 원문의 앞부분, Ollama 모델에 정해 둔 기본 생성 설정
print("지원 기능:", show_info.capabilities)
print(show_info.license[:200])
print("기본 생성 설정:", show_info.parameters)

- **지원 기능:** 라이브러리 화면의 배지와 같은 목록입니다. `thinking`·`tools`·`vision`이 있어 4-3·4-5·4-6을 쓸 수 있습니다.
- **기본 생성 설정:** `temperature 1`, `top_k 20`, `top_p 0.95`, `presence_penalty 1.5`는 모델 카드가 추론 모드에 권하는 값입니다. `presence_penalty`는 이미 나온 토큰의 점수를 깎아 같은 말의 반복을 줄입니다. `chat`에 `options`로 넘긴 값만 바뀌고, 넘기지 않은 값은 이 기본값을 씁니다.

### 4-3. 추론 모드 켜고 끄기

여러 단계를 계산해야 하는 질문으로 추론 모드를 켰을 때와 껐을 때를 비교합니다.

- **`think=True`:** 풀이 과정을 먼저 생성합니다. Ollama는 풀이 과정을 `message.thinking`에, 최종 답을 `message.content`에 나눠 돌려줍니다.
- 풀이 과정이 길어질 수 있어 `num_predict`를 넉넉히 줍니다.
- **샘플링 설정:** 추론 모드에서는 greedy(temperature 0)가 같은 말을 되풀이하기 쉬워, `temperature`를 주지 않고 모델 기본 샘플링 설정(4-2)을 씁니다. 실행할 때마다 같은 답이 나오도록 `seed`만 고정합니다.

In [ ]:
# 계산이 여러 단계인 헬프데스크 질문
license_question = [
    {"role": "system", "content": helpdesk_system},
    {"role": "user", "content": "VPN 동시 접속 라이선스가 50개입니다. 직원 120명이 오전과 오후 두 교대로 똑같이 나눠 접속하면, 교대마다 라이선스가 몇 개 모자라나요?"},
]

# 추론 모드를 켜고 모델 기본 샘플링 설정으로 생성합니다. seed로 결과를 고정합니다
thinking_response = ollama.chat(model="qwen3.5:4b-q4_K_M", messages=license_question, think=True, options={"seed": 0, "num_predict": 4096})

# 풀이 과정과 최종 답
print("풀이 과정:\n", thinking_response.message.thinking)
print("\n최종 답:\n", thinking_response.message.content)

In [ ]:
# 같은 질문을 추론 모드를 끄고 생성해 생성 토큰 수와 답을 비교합니다
no_thinking_response = ollama.chat(model="qwen3.5:4b-q4_K_M", messages=license_question, think=False, options={"temperature": 0, "num_predict": 2048})

thinking_rows = [
    {"추론 모드": "켬", "생성 토큰 수": thinking_response.eval_count, "최종 답": thinking_response.message.content},
    {"추론 모드": "끔", "생성 토큰 수": no_thinking_response.eval_count, "최종 답": no_thinking_response.message.content},
]
display(pd.DataFrame(thinking_rows))

- **생성 토큰 수:** 추론 모드를 켜면 풀이 과정까지 생성하므로 토큰 수가 늘고 그만큼 시간이 더 걸립니다.
- **정답:** 교대마다 60명이 접속하므로 라이선스가 10개 모자랍니다. 두 답을 정답과 비교합니다. 단계가 많은 계산·코드 질문일수록 추론 모드의 차이가 커집니다.
- **업무 선택:** 짧은 답이 빨리 나가야 하는 헬프데스크는 끄고, 계산·판단이 여러 단계인 업무만 켭니다. 나머지 실습은 추론 모드를 끄고 실행합니다.

### 4-4. LangChain으로 구조화된 출력

LangChain 단원에서 `ChatOpenAI`로 쓴 구조화된 출력과 도구 호출을, 채팅 모델 클래스만 `ChatOllama`로 바꿔 로컬 모델에 그대로 붙입니다.

- **`ChatOllama(model=태그, reasoning=False, temperature=0)`:** Ollama 모델을 LangChain 채팅 모델로 씁니다. `reasoning=False`는 `think=False`처럼 추론 모드를 끕니다.
- **구조화된 출력(structured output):** 모델의 답을 미리 정한 스키마(키 이름·자료형·허용 값)에 맞춰 받는 방식입니다. 헬프데스크 접수 시스템에 넣으려면 답을 정해진 키를 가진 형식으로 받아야 합니다.
- **`with_structured_output(Pydantic 클래스)`:** 클래스의 JSON 스키마를 Ollama에 넘겨 다음 토큰을 고를 때 스키마에 맞는 토큰만 남기게 하고, 받은 답을 그 클래스의 객체로 돌려줍니다.
- **`Literal[...]`:** 허용 값을 몇 개로 정합니다. 분류 결과가 정해진 범주 밖으로 나가지 않게 합니다.

In [ ]:
from typing import Literal

from langchain_ollama import ChatOllama
from pydantic import BaseModel

# Ollama의 qwen3.5:4b-q4_K_M를 LangChain 채팅 모델로 씁니다. 추론 모드를 끄고, 같은 답이 나오도록 temperature 0
llm = ChatOllama(model="qwen3.5:4b-q4_K_M", reasoning=False, temperature=0)


class HelpdeskTicket(BaseModel):
    """헬프데스크 문의를 접수 시스템에 넣을 형식으로 분류한 결과"""
    category: Literal["네트워크", "계정", "장비", "소프트웨어"]
    urgency: Literal["낮음", "보통", "높음"]
    summary: str
    first_steps: list[str]


# 모델에 넘어갈 JSON 스키마를 원형 그대로 봅니다
display(HelpdeskTicket.model_json_schema())

In [ ]:
# 접수 담당 역할과 직원 문의. 스키마의 키 뜻을 system에 함께 적어 두면 값이 더 정확해집니다
ticket_messages = [
    {"role": "system", "content": "당신은 사내 IT 헬프데스크 접수 담당입니다. 직원 문의를 분류해 category(분야), urgency(긴급도), summary(한 문장 요약), first_steps(직원이 먼저 해 볼 일 2~3개)로 답합니다."},
    {"role": "user", "content": "오늘 아침부터 회의실 프린터로 인쇄가 안 됩니다. 오후 2시 임원 회의 자료를 뽑아야 해요."},
]

# 스키마에 맞춰 답하게 하고, 답을 HelpdeskTicket 객체로 받습니다
structured_llm = llm.with_structured_output(HelpdeskTicket)
ticket = structured_llm.invoke(ticket_messages)
display(ticket)

- **형식:** 네 키가 모두 있고, `category`와 `urgency`가 허용 값 안에 있습니다. 바로 접수 시스템의 열에 넣을 수 있습니다.
- **내용:** 스키마는 형식만 보장합니다. 긴급도를 알맞게 골랐는지, 먼저 해 볼 일이 맞는지는 사람이 확인하거나 평가 단원의 방법으로 검사합니다.

### 4-5. LangChain으로 도구 호출

계정 잠금처럼 사내 시스템을 조회해야 답할 수 있는 질문은, 모델이 조회 함수를 부르게 하고 그 결과로 답하게 합니다.

- **`@tool`:** 함수 이름·docstring·인자 타입으로 도구 설명을 만듭니다.
- **`bind_tools([도구])`:** 도구 설명을 모델에 함께 보내, 모델이 답 대신 도구 호출을 요청할 수 있게 합니다.
- **`tool_calls`:** 모델이 부르겠다고 고른 도구 이름(`name`)·인자(`args`)·호출 번호(`id`)의 목록입니다. 모델은 요청만 만들고, 실행은 우리 코드가 합니다.
- **`ToolMessage`:** 도구 결과를 어느 호출(`tool_call_id`)의 결과인지와 함께 모델에 돌려주는 메시지입니다.

In [ ]:
from langchain_core.tools import tool


@tool
def check_account_lock(employee_id: str) -> str:
    """사번으로 사내 계정의 잠금 상태를 조회합니다.

    Args:
        employee_id: 직원 사번(예: E1024)
    """
    # 실습용 계정 상태 테이블입니다. 실제 서비스에서는 사내 계정 시스템을 조회합니다
    lock_status = {"E1024": "잠김(비밀번호 5회 오류, 30분 뒤 자동 해제)", "E2048": "정상"}
    return lock_status.get(employee_id, "사번을 찾을 수 없음")


# 도구를 붙인 모델은 답 대신 도구 호출을 요청할 수 있습니다
llm_with_tools = llm.bind_tools([check_account_lock])

tool_messages = [
    {"role": "system", "content": helpdesk_system},
    {"role": "user", "content": "사번 E1024입니다. 계정이 잠긴 것 같은데 확인해 주세요."},
]
ai_message = llm_with_tools.invoke(tool_messages)
display(ai_message.tool_calls)

In [ ]:
from langchain_core.messages import ToolMessage

# 모델이 요청한 도구를 우리 코드가 실행합니다
tool_call = ai_message.tool_calls[0]
tool_result = check_account_lock.invoke(tool_call["args"])
print("도구 결과:", tool_result)

# 호출 요청(AIMessage)과 도구 결과(ToolMessage)를 대화에 이어 붙이고 다시 물어 최종 답을 받습니다
tool_messages.append(ai_message)
tool_messages.append(ToolMessage(content=tool_result, tool_call_id=tool_call["id"]))
final_message = llm_with_tools.invoke(tool_messages)
print(final_message.content)

In [ ]:
from langchain.agents import create_agent

# 같은 일을 create_agent로 합니다. 호출 요청 받기, 도구 실행, 결과를 붙여 다시 묻기를 에이전트가 알아서 반복합니다
agent = create_agent(model=llm, tools=[check_account_lock], system_prompt=helpdesk_system)
agent_result = agent.invoke({"messages": [{"role": "user", "content": "사번 E1024입니다. 계정이 잠긴 것 같은데 확인해 주세요."}]})

# 마지막 메시지가 최종 답입니다
print(agent_result["messages"][-1].content)

- **호출 요청:** 모델이 질문의 사번 `E1024`를 `check_account_lock`의 인자로 넣었는지 봅니다.
- **최종 답:** 도구 결과(잠김, 30분 뒤 자동 해제)를 근거로 안내하는지 봅니다. 모델이 규정을 지어내지 않고 조회한 값으로 답하게 하는 방법입니다.

### 4-6. 이미지 입력: 사진 설명과 홍보 이미지 분석

Qwen3.5-4B는 글과 이미지를 함께 받는 모델입니다(3-5의 `image-text-to-text`). Ollama의 `qwen3.5:4b-q4_K_M`에는 이미지를 읽는 부분(비전 projector)이 함께 들어 있어, 이미지를 질문과 같이 넣을 수 있습니다. 실습 폴더 `data/images/`의 사진 한 장과 홍보 이미지 두 장을 씁니다.

- **`images=[파일 경로]`:** `ollama.chat` 메시지에 이미지 파일 경로 목록을 넣습니다. 모델은 이미지를 작은 조각으로 나눠 토큰처럼 입력에 넣습니다.
- **Transformers:** 5절의 `AutoModelForCausalLM`은 글 생성 부분만 불러오므로 이미지를 받지 못합니다. 이미지 입력은 이 절처럼 Ollama로 씁니다.

<img src="data/images/dog_park.jpg" width="500" alt="공원 잔디밭에서 골든 리트리버가 빨간 원반을 입에 물고 달려오는 사진입니다. 뒤쪽에 나무 벤치와 나무들, 멀리 걷는 사람 두 명이 보입니다.">

In [ ]:
# 사진 파일 경로를 질문과 함께 넣습니다. 이미지가 있는 폴더는 실습 폴더의 data/images
image_dir = "./data/images"

photo_response = ollama.chat(
    model="qwen3.5:4b-q4_K_M",
    messages=[{"role": "user", "content": "이 사진을 한국어로 세 문장으로 설명해 주세요. 무엇이 있고, 무엇을 하고 있고, 어디인지 적어 주세요.", "images": [f"{image_dir}/dog_park.jpg"]}],
    think=False,
    options={"temperature": 0, "num_predict": 300},
)
print("입력 토큰 수:", photo_response.prompt_eval_count)
print(photo_response.message.content)

- **입력 토큰 수:** 질문은 수십 토큰인데 입력 토큰이 수백 개입니다. 대부분이 이미지 조각입니다. 이미지가 크거나 여러 장이면 입력이 그만큼 길어지고 느려집니다.
- **설명:** 무엇이(강아지, 원반), 무엇을(달리기), 어디서(공원) 하는지 사진과 맞는지 봅니다. 작은 모델은 한국어 답에 영어 낱말이 섞이기도 합니다.

**홍보 이미지에서 상품 정보 뽑기**

마케팅팀이 모은 홍보 이미지에서 상품 정보를 테이블로 정리합니다. 4-4의 구조화된 출력에 이미지만 더하면 됩니다.

- **프롬프트 템플릿의 이미지 칸:** `ChatPromptTemplate`의 human 메시지를 목록으로 만들고, 글 조각(`{"type": "text", ...}`)과 이미지 조각(`{"type": "image_url", "image_url": "data:image/jpeg;base64,{image_base64}"}`)을 함께 둡니다. `{image_base64}` 자리에 이미지마다 다른 값을 넣어 같은 템플릿을 재사용합니다.
- **base64:** 파일 바이트를 글자로 바꾼 문자열입니다. `data:image/jpeg;base64,` 뒤에 붙여 "JPEG 이미지를 base64로 담았다"고 알립니다.
- **체인:** `promo_prompt | llm.with_structured_output(PromoInfo)`로 템플릿과 구조화된 출력 모델을 잇고, `invoke`에 템플릿 변수 dict를 넘깁니다.
- **`Field(description=...)`:** 키마다 뜻을 적어 스키마에 함께 보냅니다. 모델이 어느 값을 어느 키에 넣을지 정할 때 이 설명을 읽습니다.

<img src="data/images/cafe_promo.jpg" width="300" alt="카페 봄날의 봄 한정 신메뉴 딸기 크림 라떼 포스터입니다. 가격 5,900원, 아래 띠에 '3월 1일 ~ 3월 31일 · 2잔 주문 시 1,000원 할인'이 적혀 있습니다."> <img src="data/images/earbuds_ad.jpg" width="530" alt="무선 이어폰 소리숲 버즈 2 광고 배너입니다. 노이즈 캔슬링, 최대 30시간 재생, 출시 기념 20% 할인 159,000원이 적혀 있습니다.">

In [ ]:
import base64

from langchain_core.prompts import ChatPromptTemplate
from pydantic import Field


class PromoInfo(BaseModel):
    """홍보 이미지에서 읽은 상품 정보"""
    product_name: str = Field(description="상품 이름")
    price_won: int = Field(description="이미지에 적힌 판매 가격(원). 숫자만")
    discount: str = Field(description="할인 조건. 없으면 빈 문자열")
    period: str = Field(description="행사 기간. 없으면 빈 문자열")
    selling_points: list[str] = Field(description="이미지가 강조하는 특징")
    target_customer: str = Field(description="이미지로 보아 노리는 고객층을 한 구절로")


# 역할(system)과 질문(human)을 템플릿으로 만듭니다. human 메시지에는 글 조각과 이미지 조각을 함께 둡니다
promo_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 마케팅팀 분석 담당입니다. 홍보 이미지를 읽고 상품 정보를 정해진 형식으로 정리합니다. 이미지에 적힌 글자를 그대로 옮깁니다."),
    ("human", [
        {"type": "text", "text": "이 홍보 이미지를 분석해 주세요."},
        {"type": "image_url", "image_url": "data:image/jpeg;base64,{image_base64}"},
    ]),
])

# 템플릿과 구조화된 출력 모델을 체인으로 잇습니다
promo_chain = promo_prompt | llm.with_structured_output(PromoInfo)

promo_rows = []
for file_name in ["cafe_promo.jpg", "earbuds_ad.jpg"]:
    # 이미지 파일을 base64 문자열로 바꿉니다
    with open(f"{image_dir}/{file_name}", "rb") as f:
        image_base64 = base64.b64encode(f.read()).decode()

    # 템플릿의 {image_base64} 자리에 이번 이미지를 넣어 실행합니다
    promo = promo_chain.invoke({"image_base64": image_base64})

    row = promo.model_dump()
    row["파일"] = file_name
    promo_rows.append(row)

display(pd.DataFrame(promo_rows))

- **이미지의 글자 읽기:** 상품 이름, 가격, 할인 조건, 기간은 이미지에 적힌 글자를 읽은 값입니다. `price_won`은 `int`로 정했으므로 "5,900원" 같은 글자가 숫자로 들어옵니다. 기간이 없는 이어폰 광고는 `period`가 빈 문자열인지 봅니다.
- **추론한 값:** `target_customer`는 이미지에 적혀 있지 않아 모델이 추측한 값이고, 빈 값으로 둘 때도 있습니다. 적힌 글자를 옮긴 값과 모델이 판단한 값은 따로 확인합니다.
- **실무:** 전단·홍보물·영수증처럼 글자가 든 이미지를 테이블로 정리하는 일에 씁니다. 작은 모델은 숫자를 잘못 읽을 수 있어 가격처럼 중요한 값은 원본 이미지와 대조합니다.

## 5. Transformers로 불러와 생성하기

Ollama는 편하지만 토큰과 생성 설정을 코드로 직접 다루고 뒤 단원의 튜닝으로 이어 가려면, 같은 모델을 Transformers로 불러와야 합니다.

**GPU 메모리를 차지하는 것**

모델을 GPU에서 실행하면 가중치 말고도 여러 가지가 GPU 메모리를 나눠 씁니다. 이를 알아야 GPU 크기를 고르고 메모리 부족 오류를 피합니다.

<img src="images/교안/37_gpu_memory_parts.png" width="1050" alt="제목은 'GPU 메모리에 올라가는 것'입니다. 'GPU 메모리 24GB' 상자 안에 아래부터 CUDA 실행 환경(회색), 가장 큰 모델 가중치(파랑), KV 캐시(주황), 중간 계산값(연노랑), 점선의 남는 공간이 쌓여 있습니다. 오른쪽 칩은 차례로 '거의 고정', '파라미터 수 × 바이트', '입력과 답이 길수록 커짐', '계산하는 동안 잠깐'입니다. 왼쪽의 'Ollama 모델' 칸이 화살표로 남는 공간을 가리키고 아래에 '같은 GPU면 함께 차지'라고 적혀 있습니다. 오른쪽 아래 개발자가 상자를 가리킵니다.">

- **모델 가중치:** 층의 행렬 값(2-1)입니다. 파라미터 수 x 숫자 하나의 바이트만큼 차지하고, 모델을 올려 둔 동안 계속 남습니다(실습 모델 BF16 약 7.8GiB).
- **KV 캐시(KV cache):** 이미 처리한 토큰의 계산값(키·값)을 저장해 두는 공간입니다. 새 토큰마다 앞 토큰을 다시 계산하지 않으려고 두며, 입력과 답이 길수록 커집니다. 실습 모델은 32층 중 표준 어텐션 8층에만 쌓여 작습니다(3-5).
- **중간 계산값(활성값, activation):** 층마다 만들어 다음 층에 넘기는 계산 결과로, 계산하는 동안만 잠깐 차지합니다.
- **CUDA 실행 환경:** GPU에서 계산을 돌리는 기본 공간으로 거의 고정입니다(수백 MB~1GB).
- **다른 프로그램의 모델:** 같은 GPU를 쓰는 Ollama 모델도 차지합니다. Ollama는 KV 캐시 자리까지 미리 잡아 둡니다.
- **학습할 때만 드는 것:** 기울기와 옵티마이저 상태는 학습할 때만 듭니다(파인튜닝 단원).

아래 그림은 같은 구성을 8B 모델(Qwen3-8B)로 24GB GPU에서 측정한 예입니다.

<img src="images/교안/29_gpu_memory.png" width="1050" alt="제목은 'GPU 메모리를 차지하는 것: 8B 모델 예(Qwen3-8B, 24GB GPU 실측)'입니다. 가로 막대 세 개에 CUDA 실행 환경, 모델 가중치, KV 캐시·중간 계산값, Ollama 가중치, Ollama KV 캐시 자리를 쌓았고, 24GB GPU(23.9GiB)와 32GB GPU(31.8GiB) 선을 그었습니다. Transformers만 쓰고 짧은 대화이면 합계 약 15.7GiB, 긴 입력(8,423토큰)이면 KV 캐시·중간 계산값이 2.2GiB로 늘어 약 17.7GiB, Ollama를 내리지 않고 Transformers까지 올리면 Ollama 가중치 4.9GiB와 KV 캐시 자리 4.4GiB가 더해져 약 25.0GiB로 24GB 선을 넘습니다.">

**8B 예와 실습 모델**

- **8B 예:** 가중치 15.3GiB에 Ollama 모델(9.3GiB)이 함께 올라가면 24GB를 넘습니다.
- **실습 모델:** Qwen3.5-4B의 글 생성 부분은 가중치가 약 7.8GiB(계산값: 4,205,751,296개 x 2바이트)라 24GB GPU에 넉넉합니다. 실제 사용량은 5-3의 측정 셀로 확인합니다.

### 5-1. 모델 불러오기

다층 방어 단원에서 Llama Guard를 불러올 때 쓴 방식과 같습니다. 파일은 맨 위 미리 내려받기 셀이 받아 두었으므로 캐시에서 읽습니다.

- **`AutoModelForCausalLM`:** 3-5의 글 생성 부분(`Qwen3_5ForCausalLM`)만 불러옵니다. 이미지 입력은 4-6처럼 Ollama로 씁니다.
- **`dtype=torch.bfloat16`:** 모델 파일의 저장 정밀도(3-5의 `dtype`) 그대로 불러옵니다. 예전 코드에 보이는 `torch_dtype`은 같은 인자의 예전 이름입니다.
- **`... is falling back to its reference PyTorch implementation` 경고:** 5-3에서 처음 생성할 때 나옵니다(맥에서는 불러올 때 `The fast path is not available`). Gated DeltaNet을 빠르게 계산하는 패키지(flash-linear-attention, causal-conv1d)가 없다는 안내입니다. 느린 PyTorch 경로로 같은 결과를 냅니다.

4절에서 실행한 Ollama 모델은 마지막 요청 뒤 5분 동안 GPU에 남아 있습니다. `ollama.ps()`로 확인합니다.

- **`size_vram`:** 그 모델이 GPU 메모리에서 차지하는 바이트 수입니다.

In [ ]:
# Ollama가 지금 GPU에 올려 둔 모델 목록
display(ollama.ps())

메모리를 비워 5-3의 측정에 Ollama 몫이 섞이지 않게, Ollama 모델을 먼저 내립니다.

- **`keep_alive=0`:** 요청을 처리한 뒤 모델을 GPU에 남겨 둘 시간입니다. 0이면 바로 내립니다.

In [ ]:
# 앞에서 Ollama가 GPU에 올려 둔 모델이 있으면 내립니다. keep_alive=0은 "지금 바로 GPU에서 내리기"라는 뜻입니다
for loaded in ollama.ps().models:
    ollama.generate(model=loaded.model, keep_alive=0)
print("Ollama가 GPU에 올린 모델:", ollama.ps().models)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 토크나이저를 불러옵니다. 파일은 맨 위 미리 내려받기 셀이 받아 두었으므로 캐시에서 읽습니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Qwen3.5-4B는 이미지도 받는 모델이지만, AutoModelForCausalLM은 글 생성 부분만 불러옵니다
# 모델 파일에 저장된 정밀도(BF16) 그대로 계산 장치에 올립니다(GPU 메모리 약 8GB). 불러온 모델은 이미 평가 모드입니다
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).to(device)
print("불러온 모델:", MODEL_NAME, "| 정밀도:", model.dtype, "| 장치:", model.device)

In [ ]:
# 모델 가중치가 GPU에서 차지하는 메모리(GiB)
print("가중치:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GiB")

In [ ]:
# 불러온 모델의 층 구조를 봅니다
print(model)

# 파라미터 수를 셉니다
total_params = sum(p.numel() for p in model.parameters())
print("파라미터 수:", total_params)

- **`(0-2): 3 x Qwen3_5DecoderLayer`와 `(3): Qwen3_5DecoderLayer`:** Gated DeltaNet 블록(`linear_attn`) 3개 뒤에 어텐션 블록(`self_attn`의 `q_proj`·`k_proj`·`v_proj`·`o_proj`) 1개가 오는 묶음이 32층까지 반복됩니다(3-5). 두 블록 모두 FFN(`mlp`)을 가집니다.
- **`embed_tokens`와 `lm_head`:** 토큰 번호를 2,560 길이의 벡터로 바꾸는 임베딩 층과, 마지막 벡터를 어휘 수(248,320)만큼의 점수로 바꾸는 층입니다.
- **파라미터 수 4,205,751,296개:** 3-5의 글 생성 부분입니다. 3-4 테이블의 수에서 이미지 인코더와 다중 토큰 예측 보조층(mtp)이 빠졌습니다.

### 5-2. chat template

Instruct 모델은 학습할 때 본 대화 형식으로 입력을 받아야 제대로 답하므로, `helpdesk_messages`를 그 형식의 문자열로 바꿉니다.

- **chat template:** 대화 메시지(system·user·assistant)를 그 모델이 학습한 문자열 형식으로 바꾸는 규칙입니다. 모델마다 형식이 달라 토크나이저 파일에 함께 들어 있습니다. 다층 방어 단원에서 Llama Guard 프롬프트를 만들 때 쓴 `apply_chat_template`이 이 규칙대로 문자열을 만듭니다.
- **`enable_thinking=False`:** Qwen3.5의 chat template이 받는 인자로, 4-1의 `think=False`처럼 추론 모드를 끕니다.
- **Ollama와의 관계:** 4-1에서도 Ollama가 모델과 함께 받은 템플릿으로 대화를 이런 문자열로 바꿔 모델에 넣었습니다.

<img src="images/교안/26_generation_flow.png" width="1050" alt="제목은 'Transformers로 답을 만드는 다섯 단계'입니다. 직원이 첫 단계를 가리킵니다. 1 messages(system · user 메시지), 2 apply_chat_template(모델 형식의 문자열로), 3 tokenizer(문자열을 토큰 번호로), 4 model.generate(다음 토큰을 이어 붙임), 5 decode(새 토큰만 글자로). 아래 generate 결과: 파랑 입력 토큰 여섯 칸 뒤에 주황 새 토큰 세 칸.">

In [ ]:
# 4-1에서 Ollama에 보낸 것과 같은 helpdesk_messages를 Qwen3.5 형식의 문자열로 바꿉니다. 추론 모드는 끕니다
prompt = tokenizer.apply_chat_template(helpdesk_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print(prompt)

- **`<|im_start|>역할` ... `<|im_end|>`:** 메시지 하나의 시작과 끝을 나타내는 특수 토큰입니다. 역할 이름(system, user)이 시작 표식 바로 뒤에 붙습니다.
- **`add_generation_prompt=True`:** 문자열 끝에 `<|im_start|>assistant`를 붙여 이제 assistant가 답할 차례라고 알려 줍니다. 모델은 이 뒤를 이어 씁니다.
- **`<think>` `</think>`:** `enable_thinking=False`이면 빈 풀이 과정 칸을 미리 넣어 둡니다. 모델은 풀이를 건너뛰고 바로 답을 씁니다. `True`(기본값)이면 끝에 `<think>`만 열어 두어 모델이 풀이 과정부터 생성합니다.
- **`tokenize=False`:** 토큰 번호가 아니라 문자열로 돌려받아 눈으로 확인합니다.

### 5-3. generate로 답 만들기

문자열을 토큰 번호로 바꿔 모델에 넣고, 모델이 다음 토큰을 하나씩 이어 붙이게 합니다.

- **`tokenizer(prompt, return_tensors="pt")`:** 문자열을 PyTorch 텐서의 토큰 번호(`input_ids`)와 `attention_mask`(실제 토큰이면 1)로 바꿉니다. `.to(device)`로 모델과 같은 장치에 올립니다.
- **`max_new_tokens`:** 새로 만들 토큰의 최대 개수입니다. Ollama의 `num_predict`와 같은 뜻입니다.
- **`do_sample=False`:** 늘 가장 확률이 높은 토큰을 고르는 greedy 방식으로 생성합니다(5-4).

In [ ]:
# 문자열을 토큰 번호 텐서로 바꿔 모델과 같은 장치에 올립니다
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# greedy로 새 토큰을 200개까지 만듭니다
output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print("입력 토큰 수:", inputs["input_ids"].shape[1], "| 결과 모양:", output_ids.shape)

결과 모양의 둘째 값은 입력 토큰 수에 새로 만든 토큰 수를 더한 길이입니다. `generate`는 입력 토큰 뒤에 새 토큰을 이어 붙여 돌려줍니다.

In [ ]:
# generate 결과 원형: 입력 토큰 번호 뒤에 새 토큰 번호가 이어 붙어 있습니다
display(output_ids)

입력 길이만큼 잘라 낸 뒤 새 토큰만 글자로 바꿉니다.

- **`skip_special_tokens=True`:** `<|im_end|>` 같은 특수 토큰을 빼고 글자로 바꿉니다.

In [ ]:
# 입력 길이 뒤의 새 토큰만 남깁니다
new_ids = output_ids[0][inputs["input_ids"].shape[1]:]

# 특수 토큰을 빼고 글자로 바꿉니다
answer = tokenizer.decode(new_ids, skip_special_tokens=True)
print(answer)

In [ ]:
# 생성까지 하는 동안 PyTorch가 GPU 메모리를 가장 많이 쓴 순간의 값: 가중치 + KV 캐시 + 중간 계산값
print("생성 중 최대:", round(torch.cuda.max_memory_allocated() / 1024**3, 2), "GiB")

# GPU 전체에서 쓰고 있는 메모리: CUDA 실행 환경, 다른 프로그램(Ollama 등)까지 포함합니다
free_bytes, total_bytes = torch.cuda.mem_get_info()
print("GPU 전체 사용:", round((total_bytes - free_bytes) / 1024**3, 2), "GiB /", round(total_bytes / 1024**3, 2), "GiB")

**메모리 숫자 읽기** (실행한 GPU에 따라 값이 조금 다릅니다)
- **가중치(5-1):** 모델을 올린 직후의 `memory_allocated`로, 5절 테이블의 모델 가중치입니다. PyTorch가 GPU 메모리를 텐서마다 일정한 단위로 잡아, 계산값 약 7.83GiB보다 0.1GiB 정도 크게 나옵니다.
- **생성 중 최대:** 가중치에 생성하는 동안 생긴 KV 캐시와 중간 계산값이 더해진 값입니다. 입력과 답이 길수록 커집니다.
- **GPU 전체 사용:** PyTorch가 잡아 둔 메모리에 CUDA 실행 환경·라이브러리 작업 공간과 다른 프로그램(Ollama 등)이 쓰는 메모리까지 더한 값입니다.

같은 모델, 같은 질문인데 4-1의 Ollama 답과 문장이 조금 다를 수 있습니다. 가중치의 정밀도(Ollama 4비트 계열 Q4_K_M, Transformers BF16 원본)와 계산하는 프로그램이 달라 다음 토큰 점수가 조금씩 달라지고, Ollama의 기본 생성 설정(`presence_penalty`)도 다르기 때문입니다.

### 5-4. 생성 설정 확인과 greedy·샘플링

같은 질문에 늘 같은 답이 나올지는 생성 설정이 정하므로, 이 모델의 기본 생성 설정부터 확인합니다.

- **greedy:** 매번 확률이 가장 높은 토큰 하나를 고릅니다. 같은 입력이면 같은 답이 나옵니다.
- **샘플링:** 토큰마다 확률에 따라 무작위로 뽑습니다. 실행할 때마다 답이 달라집니다.
- **`generation_config`:** 모델 저장소의 `generation_config.json`에서 읽은 생성 기본값입니다. `generate`에 따로 넘기지 않은 설정은 이 값을 씁니다. 파일이 없으면 Transformers의 기본값을 씁니다.

<img src="images/교안/27_greedy_vs_sampling.png" width="1050" alt="제목은 '다음 토큰 고르기: greedy와 샘플링'입니다. 말풍선 'VPN이 끊기면 먼저 ___'의 후보 네트워크, 비밀번호, 재부팅(막대 길이 순). greedy (do_sample=False): 가장 긴 막대에 체크, 가장 확률이 높은 토큰, 같은 입력이면 같은 답. 샘플링 (do_sample=True): 주사위가 세 막대를 가리킴, 확률에 따라 뽑음, 실행마다 다른 답. 아래: temperature가 낮으면 1등에 몰리고, 높으면 고르게 뽑힙니다.">

In [ ]:
# 이 모델의 생성 기본값
display(model.generation_config)

- **샘플링 설정이 없음:** Qwen3.5-4B 저장소에는 `generation_config.json`이 없어 Transformers 기본값인 greedy(`do_sample=False`)로 생성합니다. 그래도 같은 답이 필요한 곳에는 5-3처럼 `do_sample=False`를 적어 두면 다른 모델로 바꿔도 같은 방식으로 생성합니다.
- **모델 카드의 권장:** 추론 모드를 끈 대화에는 `temperature=0.7`, `top_p=0.8`, `top_k=20`으로 샘플링하라고 권합니다(4-2의 Ollama 기본값은 추론 모드용입니다). 권장값의 `presence_penalty`는 Transformers `generate`에 없어 뺍니다. 이 교안은 두 실행 방법을 같은 조건으로 비교하려고 greedy를 씁니다.
- **`temperature`·`top_p`:** OpenAI API 단원에서 쓴 것과 같은 뜻입니다. **`top_k`:** 확률이 높은 상위 k개 후보 안에서만 뽑습니다.

모델 카드의 권장 샘플링 설정으로 같은 입력을 두 번 생성해 답이 달라지는지 봅니다.

In [ ]:
# 시드를 한 번 고정합니다. 이 셀을 다시 실행해도 같은 테이블이 나옵니다
torch.manual_seed(0)

# 모델 카드가 추론 모드를 끈 대화에 권하는 샘플링 설정으로 같은 입력을 두 번 생성합니다
sampled_answers = []
for run in [1, 2]:
    sample_ids = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7, top_p=0.8, top_k=20)

    # 5-3과 같이 새 토큰만 글자로 바꿉니다
    sample_new_ids = sample_ids[0][inputs["input_ids"].shape[1]:]
    sample_answer = tokenizer.decode(sample_new_ids, skip_special_tokens=True)
    sampled_answers.append({"생성 순서": run, "답": sample_answer})
display(pd.DataFrame(sampled_answers))

- 샘플링은 실행마다 다른 토큰을 뽑을 수 있어 같은 입력에도 두 답이 달라질 수 있습니다. temperature를 낮추면 확률이 높은 토큰에 몰려 두 답이 비슷해지고, 높이면 더 달라집니다.
- 헬프데스크처럼 같은 질문에 같은 안내가 나가야 하는 업무는 greedy(`do_sample=False`)나 낮은 temperature를 씁니다.

### 5-5. 대화 함수와 모델의 한계

질문 두 개를 같은 방법으로 물어보려고, 5-2·5-3의 단계를 함수 `chat` 하나로 묶습니다.

In [ ]:
def chat(question):
    """헬프데스크 안내원 역할로 질문 하나에 greedy로 답한 문자열을 돌려줍니다."""
    # system은 4-1의 헬프데스크 안내문으로 고정하고 질문만 바꿉니다
    messages = [
        {"role": "system", "content": helpdesk_system},
        {"role": "user", "content": question},
    ]

    # 추론 모드를 끈 chat template 문자열을 만들어 토큰 번호로 바꿉니다
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 같은 질문에 같은 답이 나오도록 greedy로 생성합니다
    output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)

    # 입력 길이 뒤의 새 토큰만 글자로 바꿉니다
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

In [ ]:
# 어느 회사에나 통하는 일반 질문
print(chat("노트북에서 프린터가 인쇄되지 않을 때 확인할 점을 알려 주세요."))

In [ ]:
# 우리 회사 규정을 알아야 답할 수 있는 질문
print(chat("우리 회사 사내 메일 비밀번호는 며칠마다 바꿔야 하나요?"))

- **일반 질문:** 단계 목록으로 답하는지, 내용이 맞는지 확인합니다. 내용의 정확성은 사람이 판단합니다.
- **회사 규정 질문:** 모델은 우리 회사 규정을 본 적이 없습니다. 그럴듯한 주기를 규정처럼 지어내거나 담당 부서에 문의하라고 돌려 말할 수 있습니다.

이 한계는 두 가지 방법으로 나눠 보완합니다.

- **사실(규정·수치):** 검색으로 사내 규정 문서를 찾아 프롬프트에 붙이거나(앞 단원의 RAG), 4-5처럼 사내 시스템을 조회하는 도구를 붙여 답하게 합니다.
- **답의 형식·말투·업무 지시 따르기:** 우리 도메인 예시로 모델을 튜닝해 맞춥니다(다음 단원에서 데이터셋을 만들고 이어서 튜닝).

## 6. 🖐️ 함께 따라하기: 구청 민원 안내 챗봇

본실습의 후보 테이블, Ollama, Transformers 순서를 구청 민원 안내에 적용해, 조건에 맞는 모델을 고르고 같은 질문에 답을 받아 비교합니다.

**본실습과 다른 점**
- **업무:** 구청이 민원 안내 챗봇의 운영을 위탁 업체에 맡기고, 위탁 업체가 쓸 모델을 고릅니다. 민원에는 이름·주소·연락처가 담기므로 로컬에서 실행하고, 위탁 업체가 대가를 받고 운영하므로 상업 사용이 허용된 라이선스여야 합니다.
- **후보:** 본실습의 `Qwen/Qwen3.5-4B`와 한국어 안내에 쓸 한국·외국 모델 `kakaocorp/kanana-1.5-8b-instruct-2505`, `naver-hyperclovax/HyperCLOVAX-SEED-Think-14B`, `google/gemma-3-12b-it` 네 개(3-4의 `candidate_row`로 조회)
- **질문:** system은 민원 안내원, user는 전입신고 질문입니다(5-5의 `chat` 함수는 system이 헬프데스크로 고정되어 있어 쓰지 않습니다).

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다. 완성 코드는 `정답/교안_01_정답.ipynb`의 같은 절에 있습니다.

**설정**
- **라이선스 거르기:** Apache-2.0·MIT이고 게이트가 없는 후보만 자동으로 통과시킵니다. 자체 라이선스(`other`)는 원문을 따로 확인해야 하므로 이 표에서는 통과시키지 않습니다.
- **실행 모델:** 통과한 후보 가운데 이미 내려받은 Qwen3.5-4B로 6-2·6-3을 실행합니다. 다른 후보는 같은 코드에 모델 이름만 바꿔 비교합니다.

### 6-1. 후보 테이블 만들고 거르기

In [ ]:
# 🖐️ 함께 따라하기: 후보 4개를 candidate_row로 조회해 civil_candidates 테이블을 만드세요
civil_ids = ["Qwen/Qwen3.5-4B", "kakaocorp/kanana-1.5-8b-instruct-2505", "naver-hyperclovax/HyperCLOVAX-SEED-Think-14B", "google/gemma-3-12b-it"]

civil_rows = []
for repo_id in civil_ids:
    # 1. candidate_row로 한 행(dict)을 만들어 civil_rows에 더하기
    civil_rows.append(...)

civil_candidates = pd.DataFrame(civil_rows)
display(civil_candidates)

In [ ]:
# 🖐️ 함께 따라하기: 상업 사용이 허용되는 라이선스이고 게이트가 없는 후보만 civil_ok에 남기세요
commercial_licenses = ["apache-2.0", "mit"]

# 1. 라이선스 열의 값이 commercial_licenses 안에 있는지 (isin)
license_ok = ...

# 2. 게이트 열의 값이 False인지
gate_ok = ...

civil_ok = civil_candidates[license_ok & gate_ok]
display(civil_ok)

### 6-2. Ollama로 민원 질문에 답 받기

- **`keep_alive=0`:** 5절의 Transformers 모델이 아직 GPU에 올라가 있으므로, 메모리를 비워 두려고 Ollama가 답을 준 뒤 모델을 바로 내리게 합니다.

In [ ]:
# 민원 안내원 역할(system)과 주민 질문(user)
civil_messages = [
    {"role": "system", "content": "당신은 구청 민원 안내원입니다. 주민의 질문에 한국어로 짧고 친절하게 답합니다."},
    {"role": "user", "content": "이사를 했는데 전입신고는 어디에서 어떻게 하나요?"},
]

# 🖐️ 함께 따라하기: ollama.chat으로 qwen3.5:4b-q4_K_M에 civil_messages를 보내 답을 받으세요 (think=False, keep_alive=0, temperature 0, num_predict 200)
civil_response = ...

print(civil_response.message.content)

### 6-3. Transformers로 같은 질문에 답 받기

In [ ]:
# 🖐️ 함께 따라하기: civil_messages를 추론 모드를 끈 Qwen3.5 형식의 문자열로 바꿔 greedy로 답 civil_answer를 만드세요

# 1. chat template 문자열 만들기 (tokenize=False, add_generation_prompt=True, enable_thinking=False)
civil_prompt = ...

# 2. 토큰 번호 텐서로 바꿔 계산 장치에 올리기
civil_inputs = ...

# 3. greedy(do_sample=False)로 새 토큰을 200개까지 만들기
civil_output_ids = ...

# 입력 길이 뒤의 새 토큰만 글자로 바꿉니다
civil_new_ids = civil_output_ids[0][civil_inputs["input_ids"].shape[1]:]
civil_answer = tokenizer.decode(civil_new_ids, skip_special_tokens=True)
print(civil_answer)

### 6-4. 두 답 비교하기

6-2와 6-3의 답을 읽고, 아래 두 가지를 두세 문장으로 정리합니다.

- 두 답의 형식과 내용의 차이, 그 차이가 생긴 이유
- 이 답을 그대로 주민에게 보내도 되는지에 대한 판단

**판단** (후보 테이블 결과와 두 답을 보고 두세 문장으로 쓰세요)

- ...

## 7. 핵심 정리

- **오픈 웨이트 LLM:** 가중치를 내려받아 직접 실행합니다. 데이터를 밖으로 보내지 않고 우리 데이터로 튜닝할 수 있습니다.
- **크기와 구조:** 100B를 넘는 최근 모델은 대부분 MoE이고, GPU 메모리는 전체, 계산량은 활성 파라미터 기준입니다.
- **추론 모드:** Instruct판·Thinking판으로 따로 내거나, 한 모델에서 설정으로 켜고 끕니다(하이브리드).
- **GPU 메모리:** 가중치, KV 캐시, 중간 계산값, CUDA 실행 환경, 다른 프로그램의 모델이 나눠 씁니다. 측정이 섞이지 않게 Ollama 모델을 내린 뒤 Transformers 모델을 올립니다(8B 모델은 둘을 함께 올리면 24GB를 넘습니다).
- **라이선스·벤치마크:** 라이선스는 저장소마다 확인하고, 점수는 같은 조건끼리 비교하며, 마지막 결정은 우리 질문으로 합니다.
- **후보 테이블:** `HfApi().model_info(..., expand=[...])`로 가중치를 받지 않고 라이선스·게이트·하는 일·파라미터 수를 모읍니다.
- **Ollama와 LangChain:** `ollama.chat`의 `think`로 추론 모드를 켜고 끕니다. `ChatOllama`에 `with_structured_output`·`bind_tools`를 붙이면 `ChatOpenAI`로 배운 구조화된 출력·도구 호출을 로컬 모델로 그대로 씁니다.
- **Transformers 생성 흐름:** `messages` -> `apply_chat_template`(Qwen3.5는 `enable_thinking`) -> `tokenizer` -> `model.generate` -> 입력 길이 뒤의 새 토큰만 `decode`.
- **`generation_config`와 `do_sample`:** 생성 기본값은 모델 저장소마다 다릅니다(Qwen3.5-4B는 파일이 없어 greedy). 같은 답이 필요하면 `do_sample=False`를 적고, 샘플링은 모델 카드의 권장값으로 켭니다.

교안 02에서는 정밀도와 양자화로 모델 메모리를 줄이고, Ollama의 양자화 태그(Q8_0·Q4_K_M)를 크기·속도·답으로 비교합니다.